# LeetCode 127 — Word Ladder bằng Python (Hard)

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def ladderLength(self, beginWord: str, endWord: str, wordList: List[str]) -> int:

```

Mục tiêu của bài học:

1. Hiểu **đồ thị** (graph) từ con số 0: đỉnh, cạnh, hàng xóm, danh sách kề — và đặc biệt là **đồ thị ngầm** (implicit graph): đồ thị không được cho sẵn, ta tự "sinh" cạnh khi cần.
2. Nắm chắc **BFS** (Breadth-First Search — tìm kiếm theo chiều rộng), hàng đợi **`collections.deque`**, tập **`visited`** và lý do BFS cho **đường đi ngắn nhất** trên đồ thị không trọng số.
3. Hiểu vì sao **DFS** (Depth-First Search — tìm theo chiều sâu) "thử mọi đường" vừa chậm vừa dễ sai với bài tìm đường ngắn nhất.
4. Đi qua 5 cách giải, từ trực giác đến nhanh nhất: DFS → đồ thị tường minh `O(N²·L)` → BFS thay từng ký tự `a–z` → BFS với **wildcard bucket** `"h*t"` → **BFS hai chiều** (bidirectional BFS).
5. Trace từng tầng BFS với ví dụ `"hit" → "cog"` bằng bảng và bằng hàm `trace_*`.
6. Tự viết lại lời giải, tránh lỗi kinh điển (đếm bước thay vì đếm từ, đánh dấu `visited` sai thời điểm, quên kiểm tra `endWord`...) và nhận dạng pattern **"biến đổi từng bước nhỏ, tìm số bước ít nhất"**.

> Bài anh em **#126 Word Ladder II** (liệt kê *mọi* đường ngắn nhất) được giảng ở notebook riêng. Ở đây ta chỉ cần **độ dài** của đường ngắn nhất.

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> Một **dãy biến đổi** (transformation sequence) từ từ `beginWord` đến từ `endWord` dùng từ điển `wordList` là một dãy
> `beginWord -> s1 -> s2 -> ... -> sk` sao cho:
>
> - Mỗi cặp từ **kề nhau** trong dãy khác nhau **đúng một chữ cái**.
> - Mọi `si` (với `1 <= i <= k`) đều nằm trong `wordList`. Lưu ý: `beginWord` **không cần** nằm trong `wordList`.
> - `sk == endWord`.
>
> Cho hai từ `beginWord`, `endWord` và từ điển `wordList`, hãy trả về **số từ** trong dãy biến đổi **ngắn nhất** từ `beginWord` đến `endWord`, hoặc `0` nếu không tồn tại dãy nào.

**Example 1**

```text
Input: beginWord = "hit", endWord = "cog", wordList = ["hot","dot","dog","lot","log","cog"]
Output: 5
Explanation: Một dãy biến đổi ngắn nhất là "hit" -> "hot" -> "dot" -> "dog" -> "cog", dài 5 từ.
```

**Example 2**

```text
Input: beginWord = "hit", endWord = "cog", wordList = ["hot","dot","dog","lot","log"]
Output: 0
Explanation: endWord "cog" không nằm trong wordList, nên không có dãy biến đổi hợp lệ nào.
```

### Constraints (ràng buộc) và ý nghĩa

```text
1 <= beginWord.length <= 10
endWord.length == beginWord.length
1 <= wordList.length <= 5000
wordList[i].length == beginWord.length
beginWord, endWord, và wordList[i] chỉ gồm chữ cái tiếng Anh viết thường.
beginWord != endWord
Mọi từ trong wordList đều khác nhau (unique).
```

Đọc constraints để **đoán độ phức tạp cần đạt**. Từ nay ta đặt tên:

- `N = len(wordList)` — số từ trong từ điển, tối đa **5000**.
- `L = len(beginWord)` — độ dài mỗi từ, tối đa **10**.

Hệ quả:

| Ràng buộc | Ý nghĩa cho lời giải |
|---|---|
| `N <= 5000` | `N² = 25 000 000` cặp từ — so sánh mọi cặp trong Python là **rất nặng** (dễ TLE — Time Limit Exceeded, quá thời gian). |
| `L <= 10` | Mỗi từ chỉ có `L·25 <= 250` "biến thể đổi 1 chữ" — nhỏ hơn nhiều so với `N`. Đây là chìa khóa của cách 3–5. |
| Chỉ chữ thường | Mỗi vị trí chỉ có 26 khả năng `a..z`. |
| Mọi từ cùng độ dài | Hai từ "khác 1 chữ cái" nghĩa là khác nhau **tại đúng 1 vị trí**, không có chuyện thêm/bớt ký tự. |
| `beginWord` có thể **không** có trong `wordList` | Ta phải xử lý nó như một đỉnh "ngoại lệ" làm điểm xuất phát. |
| `beginWord != endWord` | Đáp án nếu tồn tại luôn `>= 2`. |

In [1]:
from typing import List
from collections import deque
import string
import random
import time
import itertools

# Hai ví dụ của đề — dùng xuyên suốt notebook
EX1 = ("hit", "cog", ["hot", "dot", "dog", "lot", "log", "cog"])   # đáp án 5
EX2 = ("hit", "cog", ["hot", "dot", "dog", "lot", "log"])          # đáp án 0

for name, (b, e, wl) in [("EX1", EX1), ("EX2", EX2)]:
    print(name, "| begin =", b, "| end =", e, "| N =", len(wl), "| L =", len(b))
    print("     endWord có trong wordList?", e in wl)

EX1 | begin = hit | end = cog | N = 6 | L = 3
     endWord có trong wordList? True
EX2 | begin = hit | end = cog | N = 5 | L = 3
     endWord có trong wordList? False


## 2. Hiểu đề bằng tay trước khi code

### 2.1 Đếm **số từ**, không phải số bước

Dãy ngắn nhất của Example 1:

```text
hit -> hot -> dot -> dog -> cog
 1      2      3      4      5     <- đếm TỪ: 5
   (1)    (2)    (3)    (4)        <- đếm BƯỚC biến đổi: 4
```

Đề hỏi **số từ** = số bước + 1. Đây là lỗi off-by-one (lệch 1) kinh điển của bài này — ta sẽ gặp lại ở mục lỗi thường gặp.

### 2.2 "Khác đúng một chữ cái" nghĩa là gì?

So từng vị trí của hai từ cùng độ dài:

| Cặp | Vị trí 0 | Vị trí 1 | Vị trí 2 | Số vị trí khác | Kề nhau? |
|---|---|---|---|---:|:---:|
| `hit` / `hot` | h=h | **i≠o** | t=t | 1 | ✅ |
| `hot` / `dog` | **h≠d** | o=o | **t≠g** | 2 | ❌ |
| `dot` / `dot` | = | = | = | 0 | ❌ (cùng một từ) |

### 2.3 Vì sao Example 2 trả về 0?

Điều kiện "mọi `si` đều nằm trong `wordList`" áp dụng cả cho từ cuối `sk == endWord`. `"cog"` không có trong từ điển → **không** dãy nào kết thúc ở `"cog"` được → đáp án `0`, dù `"dog"` chỉ cách `"cog"` một chữ.

Cell dưới viết hai hàm nhỏ: kiểm tra hai từ kề nhau và kiểm tra một dãy có hợp lệ theo đúng 3 điều kiện của đề không. Hàm `zip(a, b)` ghép từng cặp ký tự cùng vị trí: `zip("hit", "hot")` cho `('h','h'), ('i','o'), ('t','t')`.

In [2]:
def differ_by_one(a: str, b: str) -> bool:
    # Đếm số vị trí mà hai ký tự khác nhau, kề nhau khi đúng 1 vị trí khác
    diff = 0
    for x, y in zip(a, b):
        if x != y:
            diff += 1
    return diff == 1


def is_valid_ladder(path, beginWord, endWord, wordList) -> bool:
    words = set(wordList)
    if not path or path[0] != beginWord or path[-1] != endWord:
        return False
    for w in path[1:]:                      # mọi từ sau beginWord phải thuộc từ điển
        if w not in words:
            return False
    for a, b in zip(path, path[1:]):        # từng cặp kề nhau khác đúng 1 chữ
        if not differ_by_one(a, b):
            return False
    return True


print(list(zip("hit", "hot")))
print(differ_by_one("hit", "hot"), differ_by_one("hot", "dog"), differ_by_one("dot", "dot"))

b, e, wl = EX1
print(is_valid_ladder(["hit", "hot", "dot", "dog", "cog"], b, e, wl))   # True, 5 từ
print(is_valid_ladder(["hit", "hot", "dog", "cog"], b, e, wl))          # False: hot -> dog khác 2 chữ
b, e, wl = EX2
print(is_valid_ladder(["hit", "hot", "dot", "dog", "cog"], b, e, wl))   # False: cog không có trong từ điển

[('h', 'h'), ('i', 'o'), ('t', 't')]
True False False
True
False
False


## 3. Kiến thức nền

Bài này là bài **đồ thị** đội lốt bài chuỗi. Trước khi giải, ta học lần lượt: đồ thị → đồ thị ngầm → `set` → cắt/ghép chuỗi → hàng đợi `deque` → BFS → `visited` → đệ quy & DFS.

### 3.1 Đồ thị (graph) là gì?

- **Đỉnh** (vertex / node): các "điểm", ví dụ các thành phố.
- **Cạnh** (edge): nối hai đỉnh, ví dụ con đường giữa hai thành phố. Ở bài này cạnh **vô hướng** (đi được cả hai chiều) và **không trọng số** (mọi cạnh "dài" như nhau = 1 bước).
- **Hàng xóm** (neighbor) của đỉnh `u`: các đỉnh nối trực tiếp với `u`.
- **Danh sách kề** (adjacency list): cách lưu đồ thị phổ biến nhất trong Python — một `dict` mà key là đỉnh, value là list hàng xóm.

Ví dụ đồ thị mini:

```text
    A --- B --- D
    |     |     |
    C ----+     E --- F
```

In [3]:
mini_graph = {
    "A": ["B", "C"],
    "B": ["A", "C", "D"],
    "C": ["A", "B"],
    "D": ["B", "E"],
    "E": ["D", "F"],
    "F": ["E"],
}

print("Hàng xóm của B:", mini_graph["B"])
print("D có nối với E?", "E" in mini_graph["D"])
print("Số cạnh:", sum(len(v) for v in mini_graph.values()) // 2)   # mỗi cạnh được đếm 2 lần

Hàng xóm của B: ['A', 'C', 'D']
D có nối với E? True
Số cạnh: 6


### 3.2 Đồ thị ngầm (implicit graph)

Trong Word Ladder, đề **không** đưa danh sách cạnh. Nhưng ta có thể *tưởng tượng* ra đồ thị:

- **Đỉnh** = mỗi từ (`beginWord` và các từ trong `wordList`).
- **Cạnh** giữa hai từ ⇔ hai từ khác nhau đúng 1 chữ cái.

Đồ thị kiểu này gọi là **đồ thị ngầm**: cạnh không được lưu sẵn mà được **tính ra theo một quy tắc** khi ta cần. Bài toán lúc này trở thành:

> Tìm **đường đi ngắn nhất** (ít cạnh nhất) từ đỉnh `beginWord` tới đỉnh `endWord`. Đáp án = số cạnh + 1 (= số đỉnh trên đường).

Đồ thị của Example 1 (tự vẽ bằng cách so từng cặp):

```text
                dot --- dog
               /  |      |  \
    hit --- hot   |      |   cog
               \  |      |  /
                lot --- log
```

Nhìn hình: từ `hit` chỉ có một lối là `hot`; từ `hot` rẽ hai nhánh `dot`/`lot`; cuối cùng `dog`/`log` đều nối tới `cog`. Có hai đường 5 từ: `hit-hot-dot-dog-cog` và `hit-hot-lot-log-cog`.

Cell dưới dựng đồ thị đó bằng máy (so mọi cặp) để kiểm chứng hình vẽ.

In [4]:
def build_adjacency(beginWord, wordList):
    nodes = [beginWord] + [w for w in wordList if w != beginWord]
    adj = {w: [] for w in nodes}
    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):
            if differ_by_one(nodes[i], nodes[j]):
                adj[nodes[i]].append(nodes[j])
                adj[nodes[j]].append(nodes[i])
    return adj


b, e, wl = EX1
for word, nbrs in build_adjacency(b, wl).items():
    print(f"{word}: {nbrs}")

hit: ['hot']
hot: ['hit', 'dot', 'lot']
dot: ['hot', 'dog', 'lot']
dog: ['dot', 'log', 'cog']
lot: ['hot', 'dot', 'log']
log: ['dog', 'lot', 'cog']
cog: ['dog', 'log']


### 3.3 `set`: tra cứu "có trong từ điển không?" thật nhanh

Các thuật toán của ta sẽ hỏi hàng triệu lần câu: *"từ `x` có trong `wordList` không?"*.

- Với `list`: `x in wordList` phải dò từng phần tử → `O(N)` mỗi lần hỏi.
- Với `set` (tập hợp, dùng bảng băm — hash table): `x in word_set` trung bình `O(L)` (chỉ cần băm chuỗi dài `L`), **không phụ thuộc `N`**.

Các thao tác `set` sẽ dùng:

```python
word_set = set(wordList)   # tạo set từ list
x in word_set              # kiểm tra tồn tại
word_set.remove(x)         # xóa (lỗi nếu x không có)
word_set.discard(x)        # xóa (không lỗi nếu x không có)
word_set.add(x)            # thêm
```

Lưu ý: `set` **không giữ thứ tự** chèn như `list`. Khi in để quan sát, ta sẽ dùng `sorted(...)` cho dễ đọc.

In [5]:
word_set = set(EX1[2])
print("hot" in word_set, "hit" in word_set)
word_set.remove("hot")
word_set.discard("abc")          # không có cũng không sao
print(sorted(word_set))

# So tốc độ list vs set với 5000 từ (con số cụ thể tùy máy)
big_list = [f"w{i:04d}" for i in range(5000)]
big_set = set(big_list)
t0 = time.perf_counter(); [("zzzzz" in big_list) for _ in range(2000)]; t_list = time.perf_counter() - t0
t0 = time.perf_counter(); [("zzzzz" in big_set) for _ in range(2000)]; t_set = time.perf_counter() - t0
print("list chậm hơn set khoảng", round(t_list / t_set), "lần")

True False
['cog', 'dog', 'dot', 'log', 'lot']
list chậm hơn set khoảng 590 lần


### 3.4 Thay một ký tự trong chuỗi: `word[:i] + c + word[i+1:]`

Chuỗi Python là **bất biến** (immutable): không được viết `word[1] = "o"`. Muốn "đổi chữ ở vị trí `i` thành `c`", ta **ghép** chuỗi mới:

```python
word[:i]     # phần trước vị trí i
c            # chữ mới
word[i+1:]   # phần sau vị trí i
```

Ví dụ `word = "hit"`, `i = 1`, `c = "o"`: `"h" + "o" + "t" = "hot"`.

`string.ascii_lowercase` là chuỗi `"abcdefghijklmnopqrstuvwxyz"` — duyệt nó để thử đủ 26 chữ.

Mỗi từ dài `L` có `L × 25` biến thể khác nó đúng một chữ (25 vì bỏ qua chữ hiện tại). Với `"hot"`: `3 × 25 = 75` ứng viên. Chỉ một số ít trong đó là từ có thật trong từ điển — đó chính là **hàng xóm**.

In [6]:
word = "hit"
i = 1
print(word[:i], "+", "o", "+", word[i + 1:], "=", word[:i] + "o" + word[i + 1:])
print(string.ascii_lowercase)

word = "hot"
candidates = []
for i in range(len(word)):
    for c in string.ascii_lowercase:
        if c != word[i]:
            candidates.append(word[:i] + c + word[i + 1:])
print("Số ứng viên của 'hot':", len(candidates), "| vài ứng viên đầu:", candidates[:4])

word_set = set(EX1[2]) | {"hit"}
print("Ứng viên có trong từ điển (hàng xóm):", [w for w in candidates if w in word_set])

h + o + t = hot
abcdefghijklmnopqrstuvwxyz
Số ứng viên của 'hot': 75 | vài ứng viên đầu: ['aot', 'bot', 'cot', 'dot']
Ứng viên có trong từ điển (hàng xóm): ['dot', 'lot', 'hit']


### 3.5 Hàng đợi (queue) và `collections.deque`

**Hàng đợi** hoạt động như xếp hàng mua vé: ai đến trước được phục vụ trước — **FIFO** (First In, First Out).

- Thêm vào **cuối** hàng: `append`.
- Lấy ra ở **đầu** hàng: `popleft`.

Vì sao không dùng `list`? `lst.pop(0)` phải **dịch toàn bộ** phần tử còn lại sang trái một ô → `O(n)` mỗi lần. `collections.deque` (double-ended queue — hàng đợi hai đầu) được cài đặt để `append` và `popleft` đều `O(1)`.

In [7]:
q = deque()
q.append(("hit", 1))       # (từ, số thứ tự tầng)
q.append(("hot", 2))
q.append(("dot", 3))
print("Hàng đợi:", list(q))

first = q.popleft()        # lấy người đến sớm nhất
print("Lấy ra:", first, "| còn lại:", list(q))
print("Rỗng chưa?", not q, "| độ dài:", len(q))

# list.pop(0) vs deque.popleft() với 100 000 phần tử (con số tùy máy)
lst = list(range(100_000)); t0 = time.perf_counter()
while lst: lst.pop(0)
t_list = time.perf_counter() - t0
dq = deque(range(100_000)); t0 = time.perf_counter()
while dq: dq.popleft()
t_dq = time.perf_counter() - t0
print("list.pop(0) chậm hơn deque.popleft() khoảng", round(t_list / t_dq), "lần")

Hàng đợi: [('hit', 1), ('hot', 2), ('dot', 3)]
Lấy ra: ('hit', 1) | còn lại: [('hot', 2), ('dot', 3)]
Rỗng chưa? False | độ dài: 2


list.pop(0) chậm hơn deque.popleft() khoảng 215 lần


### 3.6 BFS — tìm kiếm theo chiều rộng

Hình dung thả một hòn sỏi xuống hồ: sóng lan ra **từng vòng tròn**. BFS đi y như vậy:

- **Tầng 0** (hoặc tầng 1 nếu đếm theo số đỉnh): đỉnh xuất phát.
- **Tầng tiếp theo**: tất cả đỉnh kề với tầng trước mà **chưa từng thăm**.
- Lặp lại tới khi hết đỉnh mới.

Cài đặt bằng hàng đợi:

```text
1. Cho đỉnh xuất phát vào queue, đánh dấu đã thăm (visited).
2. Trong khi queue còn phần tử:
     u = queue.popleft()
     với mỗi hàng xóm v của u:
         nếu v chưa thăm: đánh dấu v đã thăm, đưa v vào cuối queue (tầng của v = tầng của u + 1)
```

Vì queue là FIFO, mọi đỉnh tầng `d` luôn được lấy ra **trước** mọi đỉnh tầng `d+1`. Chạy BFS trên đồ thị mini từ `A`:

In [8]:
def bfs_levels(graph, start):
    level = {start: 0}            # vừa là visited, vừa lưu khoảng cách
    q = deque([start])
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in graph[u]:
            if v not in level:    # chưa thăm
                level[v] = level[u] + 1
                q.append(v)
    return order, level


order, level = bfs_levels(mini_graph, "A")
print("Thứ tự lấy ra khỏi queue:", order)
for v in order:
    print(f"  {v}: cách A {level[v]} cạnh")

Thứ tự lấy ra khỏi queue: ['A', 'B', 'C', 'D', 'E', 'F']
  A: cách A 0 cạnh
  B: cách A 1 cạnh
  C: cách A 1 cạnh
  D: cách A 2 cạnh
  E: cách A 3 cạnh
  F: cách A 4 cạnh


### 3.7 Vì sao BFS cho đường **ngắn nhất**? Và vì sao đánh dấu `visited` **lúc đưa vào queue**?

**Ý chính**: lần đầu tiên BFS "chạm" tới một đỉnh `v` là từ một đỉnh `u` ở tầng thấp nhất có thể (vì các tầng được xử lý theo thứ tự tăng dần 0, 1, 2, ...). Nên `level[v] = level[u] + 1` chính là khoảng cách ngắn nhất. Mọi lần chạm `v` về sau đều đến từ tầng bằng hoặc cao hơn → không bao giờ tốt hơn → **bỏ qua được**. Đây là lý do một đỉnh chỉ cần thăm **một lần**.

Ở mini graph: `D` cách `A` 2 cạnh (`A-B-D`), dù còn đường dài hơn `A-C-B-D`.

**Thời điểm đánh dấu**: nếu chỉ đánh dấu khi **lấy ra** (`popleft`), một đỉnh có thể bị **đưa vào queue nhiều lần** trước khi kịp được lấy ra (mỗi hàng xóm ở tầng trước đẩy nó vào một lần). Kết quả vẫn đúng nhưng queue phình to, tốn thời gian. Đánh dấu ngay khi **đưa vào** đảm bảo mỗi đỉnh vào queue đúng một lần.

Cell dưới đếm số lần `append` của hai kiểu trên một đồ thị "dày" (mọi cặp đỉnh đều nối nhau — đồ thị đầy đủ 30 đỉnh).

In [9]:
def count_pushes(graph, start, mark_on_push=True):
    visited = {start}
    q = deque([start])
    pushes = 1
    while q:
        u = q.popleft()
        if not mark_on_push:
            visited.add(u)              # kiểu sai: chỉ đánh dấu khi lấy ra
        for v in graph[u]:
            if v not in visited:
                if mark_on_push:
                    visited.add(v)      # kiểu đúng: đánh dấu ngay khi đưa vào
                q.append(v)
                pushes += 1
    return pushes


complete = {i: [j for j in range(30) if j != i] for i in range(30)}
print("Đánh dấu lúc đưa vào :", count_pushes(complete, 0, True), "lần append")
print("Đánh dấu lúc lấy ra  :", count_pushes(complete, 0, False), "lần append")

Đánh dấu lúc đưa vào : 30 lần append
Đánh dấu lúc lấy ra  : 436 lần append


### 3.8 Đệ quy (recursion) và DFS — tìm theo chiều sâu

**Đệ quy**: một hàm tự gọi lại chính nó với bài toán nhỏ hơn, và có **điều kiện dừng** (base case). Mỗi lời gọi được đặt lên **call stack** (ngăn xếp lời gọi); gọi xong thì quay về đúng chỗ cũ.

**DFS** đi theo kiểu "đâm thẳng một nhánh tới cùng, cụt thì **quay lui** (backtrack) thử nhánh khác". Muốn liệt kê **mọi đường đi đơn** (không lặp đỉnh), ta giữ tập `on_path` gồm các đỉnh đang nằm trên đường hiện tại:

```text
dfs(u):
    nếu u là đích: ghi nhận đường đi
    với mỗi hàng xóm v không nằm trên đường hiện tại:
        thêm v vào đường    <- "đi tới"
        dfs(v)
        bỏ v khỏi đường     <- "quay lui", để nhánh khác còn dùng được v
```

Liệt kê mọi đường từ `A` tới `D` trong mini graph:

In [10]:
def all_simple_paths(graph, start, goal):
    paths = []
    path = [start]
    on_path = {start}

    def dfs(u):
        if u == goal:
            paths.append(list(path))       # chép lại đường hiện tại
            return
        for v in graph[u]:
            if v not in on_path:
                path.append(v); on_path.add(v)        # đi tới
                dfs(v)
                path.pop(); on_path.remove(v)         # quay lui

    dfs(start)
    return paths


for p in all_simple_paths(mini_graph, "A", "D"):
    print(" -> ".join(p), f"({len(p) - 1} cạnh)")

A -> B -> D (2 cạnh)
A -> C -> B -> D (3 cạnh)


Quan sát: DFS tìm thấy `A -> B -> D` trước, nhưng đó là **nhờ may mắn** của thứ tự hàng xóm. Nếu `mini_graph["A"]` liệt kê `C` trước `B`, đường đầu tiên DFS gặp sẽ là `A -> C -> B -> D` (3 cạnh) — **không** phải ngắn nhất. DFS không có khái niệm "tầng", nên muốn chắc chắn có đường ngắn nhất, nó phải **thử hết** mọi đường. Ghi nhớ điểm này cho Cách 1.

## 4. Cách 1 — DFS thử mọi đường (cách người mới hay nghĩ đầu tiên)

**Suy nghĩ tự nhiên**: "Từ `beginWord`, thử đổi sang mọi từ kề nhau, rồi từ đó lại thử tiếp... Khi tới `endWord` thì ghi lại độ dài, giữ độ dài nhỏ nhất."

Đây chính là DFS liệt kê mọi đường đơn ở mục 3.8, áp dụng cho đồ thị ngầm:

- Hàng xóm của `word` = các từ `x` trong `wordList` chưa nằm trên đường và `differ_by_one(word, x)`.
- `count` = số từ trên đường hiện tại.
- Biến `calls` đếm số lời gọi đệ quy để thấy chi phí.

In [11]:
def ladder_dfs_all_paths(beginWord: str, endWord: str, wordList: List[str], max_calls=None):
    if endWord not in wordList:
        return 0, 0
    best = 0
    calls = 0
    on_path = {beginWord}

    def dfs(word, count):
        nonlocal best, calls
        calls += 1
        if max_calls is not None and calls > max_calls:
            return                                  # chốt chặn để notebook không treo
        if word == endWord:
            if best == 0 or count < best:
                best = count
            return
        for x in wordList:
            if x not in on_path and differ_by_one(word, x):
                on_path.add(x)
                dfs(x, count + 1)
                on_path.remove(x)

    dfs(beginWord, 1)
    return best, calls


print(ladder_dfs_all_paths(*EX1))   # (đáp án, số lời gọi)
print(ladder_dfs_all_paths(*EX2))

(5, 24)
(0, 0)


Cách 1 cho đáp án đúng với hai ví dụ. Vậy vướng ở đâu? Có **hai** vấn đề.

### Vấn đề 1: đường đầu tiên DFS tìm thấy thường **không** ngắn nhất

Giả sử ta muốn "ăn gian": dừng ngay khi tới `endWord` lần đầu. Thử cùng từ điển của Example 1 nhưng **đổi thứ tự** liệt kê: `["hot","lot","dot","dog","log","cog"]`. DFS đi `hot → lot`, rồi từ `lot` thấy `dot` trước (vì `dot` đứng trước `log` trong list), từ `dog` thấy `log` trước `cog`... và lòng vòng thành đường **7 từ**.

In [12]:
def dfs_first_path(beginWord, endWord, wordList):
    path = [beginWord]
    on_path = {beginWord}

    def dfs(word):
        if word == endWord:
            return True                  # dừng ngay lần đầu tới đích
        for x in wordList:
            if x not in on_path and differ_by_one(word, x):
                path.append(x); on_path.add(x)
                if dfs(x):
                    return True
                path.pop(); on_path.remove(x)
        return False

    return list(path) if dfs(beginWord) else []


reordered = ["hot", "lot", "dot", "dog", "log", "cog"]
p = dfs_first_path("hit", "cog", reordered)
print(" -> ".join(p), "|", len(p), "từ  (ngắn nhất là 5)")

hit -> hot -> lot -> dot -> dog -> log -> cog | 7 từ  (ngắn nhất là 5)


### Vấn đề 2: muốn đúng phải thử **mọi** đường → số đường bùng nổ theo hàm mũ

Có thể bạn nghĩ: "Dùng `visited` **vĩnh viễn** như BFS để khỏi thăm lại." Sai! Với DFS, lần đầu ta thăm một từ có thể là qua một đường **dài**; đánh dấu vĩnh viễn sẽ chặn mất đường **ngắn** đi qua từ đó về sau. Hàm `dfs_global_visited` dưới đây chứng minh: nó trả về **7** thay vì **5**.

Vì vậy DFS đúng buộc phải **quay lui** (bỏ đánh dấu khi rút về) — tức là duyệt mọi **đường đi đơn**. Số đường đơn trong đồ thị có thể lên tới cỡ `N!` (giai thừa).

Thí nghiệm: từ điển gồm **mọi** từ dài `L` chỉ dùng chữ `a`, `b` (đồ thị "hình lập phương" `L` chiều), đi từ `"aa..a"` tới `"bb..b"`.

In [13]:
def dfs_global_visited(beginWord, endWord, wordList):
    # SAI: đánh dấu visited vĩnh viễn, không quay lui
    visited = {beginWord}
    best = 0

    def dfs(word, count):
        nonlocal best
        if word == endWord:
            best = count if best == 0 else min(best, count)
            return
        for x in wordList:
            if x not in visited and differ_by_one(word, x):
                visited.add(x)
                dfs(x, count + 1)        # không có visited.remove(x)

    dfs(beginWord, 1)
    return best


print("DFS visited vĩnh viễn:", dfs_global_visited("hit", "cog", reordered), "(sai, phải là 5)")

for L in [3, 4, 5]:
    words = ["".join(t) for t in itertools.product("ab", repeat=L)]
    ans, calls = ladder_dfs_all_paths("a" * L, "b" * L, words[1:], max_calls=300_000)
    note = "  <- đã CHẠM CHỐT 300 000 lời gọi, chưa xong!" if calls > 300_000 else ""
    print(f"L={L}: N={len(words) - 1:2d} từ, số lời gọi DFS = {calls:,}{note}")

DFS visited vĩnh viễn: 7 (sai, phải là 5)
L=3: N= 7 từ, số lời gọi DFS = 52
L=4: N=15 từ, số lời gọi DFS = 25,013


L=5: N=31 từ, số lời gọi DFS = 300,041  <- đã CHẠM CHỐT 300 000 lời gọi, chưa xong!


Chỉ với `N = 31` từ (`L = 5`), DFS đã vượt 300 000 lời gọi mà chưa xong (khi chạy thử không giới hạn, nó chạy hơn 2 phút vẫn chưa xong). Đề cho `N` tới 5000 → DFS **vô vọng**.

**Độ phức tạp Cách 1**: thời gian tỉ lệ với số đường đi đơn — hàm mũ, tệ nhất cỡ `O(N! · N · L)` (mỗi lời gọi còn tốn `O(N·L)` để dò hàng xóm). Bộ nhớ `O(N)` cho đường đi và call stack.

**Nút thắt**: DFS không biết "khoảng cách". Ta cần một cách duyệt mà **đỉnh nào gần hơn được thăm trước** — đó là BFS.

## 5. Bước đột phá: đường ngắn nhất trên đồ thị không trọng số = BFS theo tầng

Mọi cạnh có "độ dài" bằng nhau (1 lần đổi chữ). Khi đó BFS từ `beginWord` chia các từ thành từng **tầng**, và **tầng của `endWord` chính là đáp án** (nếu ta đánh số tầng bắt đầu từ 1 = số từ).

Bảng tầng của Example 1 (tự làm bằng tay, dựa vào hình đồ thị mục 3.2):

| Tầng (= số từ tính tới đây) | Các từ trong tầng | Đến từ |
|---:|---|---|
| 1 | `hit` | xuất phát |
| 2 | `hot` | `hit` |
| 3 | `dot`, `lot` | `hot` |
| 4 | `dog`, `log` | `dot` → `dog`, `lot` → `log` |
| 5 | `cog` | `dog` (hoặc `log`) |

`cog` ở tầng 5 → đáp án **5**. Không cần thử từng đường: BFS tự động "biết" khoảng cách nhờ thứ tự tầng.

Câu hỏi còn lại chỉ là: **tìm hàng xóm của một từ bằng cách nào cho nhanh?** Cách 2, 3, 4 là ba câu trả lời khác nhau cho câu hỏi đó; Cách 5 giảm số từ phải duyệt.

## 6. Cách 2 — Dựng đồ thị tường minh bằng so sánh mọi cặp, rồi BFS

**Ý tưởng**: biến đồ thị ngầm thành đồ thị **tường minh** (explicit — lưu hẳn danh sách kề) bằng hàm `build_adjacency` ở mục 3.2 (so mọi cặp từ), sau đó chạy BFS chuẩn ở mục 3.6.

Các bước:

1. Nếu `endWord` không có trong `wordList` → trả `0`.
2. Dựng `adj` từ `beginWord` + `wordList`.
3. BFS từ `beginWord` với `dist[beginWord] = 1`. Khi lấy ra `endWord` → trả `dist[endWord]`.
4. Queue cạn mà chưa gặp → trả `0`.

In [14]:
def ladder_explicit_graph(beginWord: str, endWord: str, wordList: List[str]) -> int:
    if endWord not in wordList:
        return 0
    adj = build_adjacency(beginWord, wordList)          # O(N² · L)
    dist = {beginWord: 1}                               # dist = số từ tính tới đây
    q = deque([beginWord])
    while q:
        word = q.popleft()
        if word == endWord:
            return dist[word]
        for nxt in adj[word]:
            if nxt not in dist:
                dist[nxt] = dist[word] + 1
                q.append(nxt)
    return 0


print(ladder_explicit_graph(*EX1))   # 5
print(ladder_explicit_graph(*EX2))   # 0
print(ladder_explicit_graph("hit", "cog", reordered))   # thứ tự list không ảnh hưởng BFS: 5

5
0
5


### Nút thắt của Cách 2: bước dựng đồ thị `O(N²·L)`

Với `N + 1` đỉnh, số cặp phải so là `(N+1)·N / 2`. Mỗi cặp so `L` ký tự.

Với `N = 5000`, `L = 10`: khoảng **12,5 triệu cặp × 10 ký tự ≈ 125 triệu phép so sánh ký tự** — trong Python mất **vài giây** (mục 10.5 đo thực tế), dễ TLE. Phần BFS phía sau thì rẻ (`O(N + E)`, `E` = số cạnh), nhưng ta đã "chết" từ bước dựng đồ thị.

Ngoài ra danh sách kề có thể chứa tới `O(N²)` cạnh (khi rất nhiều từ kề nhau) → tốn bộ nhớ.

**Quan sát then chốt**: một từ chỉ có tối đa `25·L ≤ 250` khả năng hàng xóm, nhưng Cách 2 lại hỏi cả `N = 5000` từ "mày có phải hàng xóm tao không?". Hãy **đảo câu hỏi**: tự sinh 250 ứng viên rồi tra `set`. Cell dưới đếm cụ thể hai con số này, rồi đo thời gian dựng đồ thị với `N` = 500, 1000, 2000: hãy để ý mỗi lần `N` **gấp đôi** thì thời gian tăng khoảng **4 lần** — dấu hiệu của `O(N²)`.

In [15]:
N, L = 5000, 10
pairs = (N + 1) * N // 2
print(f"Cách 2: {pairs:,} cặp × {L} ký tự = {pairs * L:,} phép so ký tự")
print(f"Ý tưởng mới: {N:,} từ × {L} vị trí × 25 chữ = {N * L * 25:,} ứng viên cần tra set")

# Đo thực tế bước dựng đồ thị với N nhỏ hơn để thấy tốc độ tăng theo N² (con số tùy máy)
rng = random.Random(127)
for n in [500, 1000, 2000]:
    wl = sorted({"".join(rng.choice("abcdefghij") for _ in range(5)) for _ in range(n * 2)})[:n]
    t0 = time.perf_counter(); build_adjacency("aaaaa", wl); t = time.perf_counter() - t0
    print(f"  N={n:5d}: dựng đồ thị mất {t:.2f}s")

Cách 2: 12,502,500 cặp × 10 ký tự = 125,025,000 phép so ký tự
Ý tưởng mới: 5,000 từ × 10 vị trí × 25 chữ = 1,250,000 ứng viên cần tra set
  N=  500: dựng đồ thị mất 0.07s


  N= 1000: dựng đồ thị mất 0.19s


  N= 2000: dựng đồ thị mất 0.67s


## 7. Cách 3 — BFS, sinh hàng xóm bằng cách thay từng ký tự `a–z`

**Mấu chốt cải tiến** so với Cách 2: không dựng đồ thị trước. Mỗi khi lấy một từ `word` ra khỏi queue, ta **sinh** mọi biến thể đổi 1 chữ (`L × 25` chuỗi, kỹ thuật ở mục 3.4) và giữ lại những chuỗi có trong `word_set`. Đây đúng là tinh thần **đồ thị ngầm**: cạnh được tính "tại chỗ".

**Mẹo `visited` gọn**: thay vì giữ một tập `visited` riêng, ta **xóa từ khỏi `word_set`** ngay khi đưa nó vào queue. Một từ đã xóa sẽ không bao giờ được sinh lại như hàng xóm → tương đương "đã thăm". (`beginWord` cũng được `discard` phòng khi nó có trong `wordList`.)

```text
word_set = set(wordList); nếu endWord không có → return 0
queue = [(beginWord, 1)]; xóa beginWord khỏi word_set
trong khi queue còn:
    word, level = popleft()
    nếu word == endWord: return level
    với mỗi vị trí i, mỗi chữ c ≠ word[i]:
        nxt = word[:i] + c + word[i+1:]
        nếu nxt trong word_set:
            xóa nxt khỏi word_set        # đánh dấu đã thăm lúc ĐƯA VÀO
            append (nxt, level + 1)
return 0
```

In [16]:
def ladder_bfs_26(beginWord: str, endWord: str, wordList: List[str]) -> int:
    word_set = set(wordList)
    if endWord not in word_set:
        return 0
    word_set.discard(beginWord)
    q = deque([(beginWord, 1)])
    while q:
        word, level = q.popleft()
        if word == endWord:
            return level
        for i in range(len(word)):
            prefix, suffix = word[:i], word[i + 1:]
            for c in string.ascii_lowercase:
                if c == word[i]:
                    continue
                nxt = prefix + c + suffix
                if nxt in word_set:
                    word_set.remove(nxt)        # visited
                    q.append((nxt, level + 1))
    return 0


print(ladder_bfs_26(*EX1))   # 5
print(ladder_bfs_26(*EX2))   # 0

5
0


### Trace Cách 3 với `hit → cog`

Bảng dưới ghi mỗi lần `popleft`: từ lấy ra, tầng của nó, các hàng xóm **mới** tìm được (theo thứ tự vị trí `i` rồi chữ `a→z`), và queue sau bước đó. Cột cuối là các từ **còn lại** trong `word_set` (chưa thăm).

| Bước | Lấy ra (tầng) | Hàng xóm mới | Queue sau bước | `word_set` còn lại |
|---:|---|---|---|---|
| 1 | `hit` (1) | `hot` | `hot(2)` | cog, dog, dot, log, lot |
| 2 | `hot` (2) | `dot`, `lot` | `dot(3) lot(3)` | cog, dog, log |
| 3 | `dot` (3) | `dog` | `lot(3) dog(4)` | cog, log |
| 4 | `lot` (3) | `log` | `dog(4) log(4)` | cog |
| 5 | `dog` (4) | `cog` | `log(4) cog(5)` | (rỗng) |
| 6 | `log` (4) | — (`cog` đã bị xóa) | `cog(5)` | (rỗng) |
| 7 | `cog` (5) | **gặp `endWord` → return 5** | | |

Chú ý bước 3: từ `dot` có thể đổi thành `lot` hay `hot`, nhưng hai từ này **đã bị xóa** khỏi `word_set` nên không bị thêm lại. Bước 6: `log` cũng kề `cog` nhưng `cog` đã được `dog` "nhận" trước — cùng tầng 5, nên không mất gì.

Hàm `trace_bfs_26` in đúng các thông tin này:

In [17]:
def trace_bfs_26(beginWord, endWord, wordList):
    word_set = set(wordList)
    if endWord not in word_set:
        print("endWord không có trong từ điển -> 0"); return 0
    word_set.discard(beginWord)
    q = deque([(beginWord, 1)])
    step = 0
    while q:
        word, level = q.popleft()
        step += 1
        if word == endWord:
            print(f"Bước {step}: lấy {word}({level}) == endWord -> return {level}")
            return level
        found = []
        for i in range(len(word)):
            for c in string.ascii_lowercase:
                nxt = word[:i] + c + word[i + 1:]
                if c != word[i] and nxt in word_set:
                    word_set.remove(nxt); q.append((nxt, level + 1)); found.append(nxt)
        qs = " ".join(f"{w}({d})" for w, d in q)
        print(f"Bước {step}: lấy {word}({level}) | mới: {found or '-'} | queue: {qs} | còn: {sorted(word_set)}")
    print("Queue cạn -> 0"); return 0


trace_bfs_26(*EX1)

Bước 1: lấy hit(1) | mới: ['hot'] | queue: hot(2) | còn: ['cog', 'dog', 'dot', 'log', 'lot']
Bước 2: lấy hot(2) | mới: ['dot', 'lot'] | queue: dot(3) lot(3) | còn: ['cog', 'dog', 'log']
Bước 3: lấy dot(3) | mới: ['dog'] | queue: lot(3) dog(4) | còn: ['cog', 'log']
Bước 4: lấy lot(3) | mới: ['log'] | queue: dog(4) log(4) | còn: ['cog']
Bước 5: lấy dog(4) | mới: ['cog'] | queue: log(4) cog(5) | còn: []
Bước 6: lấy log(4) | mới: - | queue: cog(5) | còn: []
Bước 7: lấy cog(5) == endWord -> return 5


5

### Nhận xét và độ phức tạp Cách 3

- **Kiểm tra đích lúc lấy ra** (như trên) dễ chứng minh đúng: khi `popleft` một từ ở tầng `d`, `d` chắc chắn là khoảng cách ngắn nhất. Có thể kiểm tra sớm hơn ngay lúc **sinh** `nxt == endWord` để tiết kiệm phần còn lại của tầng — kết quả không đổi (Cách 5 sẽ dùng kiểu kiểm tra sớm này).
- Mỗi từ vào queue **tối đa một lần** (nhờ xóa khỏi `word_set`).

Độ phức tạp (với `N = len(wordList)`, `L = len(beginWord)`):

- **Thời gian `O(N · L · 26 · L) = O(26 · N · L²)`**: mỗi từ lấy ra sinh `25·L` ứng viên; mỗi ứng viên tốn `O(L)` để ghép chuỗi và băm khi tra `set`. Với `N = 5000, L = 10`: cỡ `5000 · 250 · 10 ≈ 12,5 triệu` phép cơ bản, nhanh hơn Cách 2 khoảng 10 lần và quan trọng hơn: **tuyến tính theo `N`** thay vì bình phương.
- **Bộ nhớ `O(N · L)`**: `word_set` và queue chứa tối đa `N` từ dài `L`.

Điểm yếu còn lại: thử cả 26 chữ ở mỗi vị trí, kể cả khi từ điển rất "thưa" (đa số ứng viên không tồn tại). Cách 4 khắc phục bằng cách nhóm sẵn các từ có chung "khuôn".

## 8. Cách 4 — BFS với wildcard bucket `"h*t"`

**Ý tưởng**: hai từ kề nhau ⇔ khi che (thay bằng `*`) **cùng một vị trí** thì chúng thành **cùng một khuôn** (pattern).

- `hot` có 3 khuôn: `*ot`, `h*t`, `ho*`.
- `dot` có 3 khuôn: `*ot`, `d*t`, `do*`.
- `hot` và `dot` chung khuôn `*ot` → kề nhau.

Ta tiền xử lý một `dict` gọi là **bucket** (cái xô): `bucket[pattern] = list các từ có khuôn đó`. Hàng xóm của `word` = hợp của các bucket ứng với `L` khuôn của `word` (bỏ chính nó).

Kỹ thuật Python mới: **`dict.setdefault(key, [])`** trả về list ở `key`, tạo list rỗng nếu chưa có — để viết gọn `bucket.setdefault(p, []).append(w)`. (Có thể dùng `collections.defaultdict(list)` cho cùng mục đích.)

So với Cách 3: thay vì sinh `25·L` chuỗi ứng viên rồi tra, mỗi từ chỉ sinh `L` khuôn và nhận **ngay** danh sách hàng xóm thật.

In [18]:
def build_buckets(wordList):
    bucket = {}
    for w in wordList:
        for i in range(len(w)):
            pattern = w[:i] + "*" + w[i + 1:]
            bucket.setdefault(pattern, []).append(w)
    return bucket


buckets = build_buckets(EX1[2])
print("Số bucket:", len(buckets))
for p, ws in buckets.items():
    if len(ws) >= 2:                        # chỉ in bucket có từ 2 từ trở lên (tức là có cạnh)
        print(f"  {p}: {ws}")
print("Khuôn của 'hit':", ["hit"[:i] + "*" + "hit"[i + 1:] for i in range(3)])

Số bucket: 12
  *ot: ['hot', 'dot', 'lot']
  do*: ['dot', 'dog']
  *og: ['dog', 'log', 'cog']
  lo*: ['lot', 'log']
Khuôn của 'hit': ['*it', 'h*t', 'hi*']


### BFS trên bucket và mẹo **xóa bucket sau khi dùng**

BFS giống Cách 3, chỉ đổi phần "tìm hàng xóm":

```text
với mỗi khuôn p của word:
    với mỗi nxt trong bucket.get(p, []):
        nếu nxt chưa thăm: đánh dấu, đưa vào queue
    bucket[p] = []        # <- mẹo quan trọng
```

**Vì sao xóa bucket?** Giả sử 1000 từ cùng khuôn `"a*"`... (ví dụ `"aa","ab",...` với `L=2`). Từ đầu tiên lấy ra đã đẩy *hết* các từ trong bucket vào queue. Nếu không xóa, mỗi từ trong 1000 từ đó khi được lấy ra lại duyệt **lại** đúng bucket 1000 phần tử → `10⁶` lần kiểm tra vô ích. Khi một bucket đã được duyệt một lần, mọi từ trong nó đã thăm rồi → duyệt lại không bao giờ có ích → xóa được. Nhờ vậy **mỗi bucket chỉ được duyệt một lần**.

Ở đây ta dùng `visited` là một `set` riêng (vì `wordList` nằm trong bucket, không xóa khỏi `word_set` như Cách 3).

In [19]:
def ladder_wildcard(beginWord: str, endWord: str, wordList: List[str], trace=False) -> int:
    if endWord not in wordList:
        return 0
    bucket = build_buckets(wordList)                   # O(N · L²)
    visited = {beginWord}
    q = deque([(beginWord, 1)])
    while q:
        word, level = q.popleft()
        if word == endWord:
            if trace: print(f"lấy {word}({level}) == endWord -> return {level}")
            return level
        found = []
        for i in range(len(word)):
            p = word[:i] + "*" + word[i + 1:]
            for nxt in bucket.get(p, []):
                if nxt not in visited:
                    visited.add(nxt)
                    q.append((nxt, level + 1))
                    found.append(f"{nxt} qua {p}")
            bucket[p] = []                             # bucket đã dùng xong
        if trace: print(f"lấy {word}({level}) | mới: {found or '-'}")
    return 0


ladder_wildcard(*EX1, trace=True)
print(ladder_wildcard(*EX2))

lấy hit(1) | mới: ['hot qua h*t']
lấy hot(2) | mới: ['dot qua *ot', 'lot qua *ot']
lấy dot(3) | mới: ['dog qua do*']
lấy lot(3) | mới: ['log qua lo*']
lấy dog(4) | mới: ['cog qua *og']
lấy log(4) | mới: -
lấy cog(5) == endWord -> return 5
0


Output trace khớp bảng tầng ở mục 5: `hot` (tầng 2) đến từ khuôn `h*t`; `dot`, `lot` (tầng 3) đến từ khuôn `*ot`; `dog` qua `do*`, `log` qua `lo*`; `cog` qua `*og`.

### Độ phức tạp Cách 4

- **Dựng bucket**: `N` từ × `L` khuôn × `O(L)` để tạo chuỗi khuôn = `O(N · L²)`.
- **BFS**: mỗi từ lấy ra tạo `L` khuôn (`O(L²)`), và nhờ xóa bucket, tổng số lần duyệt phần tử trong các bucket ≤ tổng kích thước bucket = `N · L`; mỗi lần tra `visited` `O(L)`. Tổng **`O(N · L²)`** — bỏ được hệ số 26 của Cách 3.
- **Bộ nhớ `O(N · L²)`**: có tới `N·L` khuôn, mỗi khuôn là chuỗi dài `L`, cộng các list trong bucket.

Cách 3 và Cách 4 có cùng "hình dạng" BFS; chúng chỉ khác cách tìm hàng xóm. Cả hai vẫn **lan sóng từ một phía**. Bước cải tiến lớn tiếp theo là thay đổi chính hình dạng của cuộc tìm kiếm.

## 9. Cách 5 — BFS hai chiều (bidirectional BFS) — cách nhanh nhất

### 9.1 Trực giác: hai làn sóng gặp nhau ở giữa

BFS một chiều lan từ `beginWord` cho tới khi chạm `endWord`. Nếu mỗi từ có trung bình `b` hàng xóm mới (gọi là **hệ số rẽ nhánh** — branching factor) và đáp án cách `d` bước, số từ phải thăm cỡ `b + b² + ... + b^d ≈ b^d`.

BFS **hai chiều** thả **hai** hòn sỏi: một ở `beginWord`, một ở `endWord`. Hai làn sóng lan về phía nhau và **gặp nhau ở giữa**, mỗi bên chỉ cần đi khoảng `d/2` bước → tổng cỡ `2 · b^(d/2)`.

Vì sao đi ngược từ `endWord` được? Quan hệ "khác đúng 1 chữ" là **đối xứng**: `a` kề `b` thì `b` kề `a` — đồ thị vô hướng.

Ví dụ số: `b = 10`, `d = 6`:

In [20]:
b, d = 10, 6
one_way = sum(b ** k for k in range(1, d + 1))
two_way = 2 * sum(b ** k for k in range(1, d // 2 + 1))
print(f"BFS một chiều : khoảng {one_way:,} từ")
print(f"BFS hai chiều : khoảng {two_way:,} từ")
print(f"Tiết kiệm     : khoảng {one_way // two_way} lần")

BFS một chiều : khoảng 1,111,110 từ
BFS hai chiều : khoảng 2,220 từ
Tiết kiệm     : khoảng 500 lần


### 9.2 Thuật toán chi tiết

Ta không dùng `deque` mà dùng hai **`set`** làm "mặt sóng" (frontier):

- `front`: tầng hiện tại của phía đang mở rộng.
- `back`: tầng hiện tại của phía bên kia.
- `words`: các từ **chưa thăm** bởi phía nào cả (ban đầu là `wordList` bỏ `beginWord` và `endWord`).
- `length`: bộ đếm giữ **bất biến** `length = d_front + d_back + 1`, với `d_front`, `d_back` là số bước mà mỗi phía đã đi. Ban đầu cả hai là 0 → `length = 1`. Ý nghĩa: nếu một từ ở `front` kề một từ ở `back`, đường hoàn chỉnh có `d_front + 1` từ (phía front) cộng `d_back + 1` từ (phía back) = `length + 1` từ.

Mỗi vòng:

1. **Luôn mở rộng phía nhỏ hơn**: nếu `len(front) > len(back)` thì đổi chỗ `front, back = back, front`. Mặt sóng nhỏ sinh ít ứng viên hơn — đây là nửa còn lại của sức mạnh bidirectional.
2. Với mỗi `word` trong `front`, sinh mọi biến thể `nxt` (như Cách 3):
   - **Nếu `nxt` nằm trong `back`**: hai sóng chạm nhau! Đường đi = (`d_front + 1` từ phía front) + (`d_back + 1` từ phía back) = `length + 1` từ → **return `length + 1`**.
   - Ngược lại nếu `nxt` trong `words`: xóa khỏi `words`, thêm vào `next_front`.
3. `front = next_front`, `length += 1` (phía vừa mở rộng đi thêm 1 bước).
4. Nếu một phía rỗng → hai sóng không bao giờ gặp → `0`.

⚠️ Hai chỗ bắt buộc:

- **Phải kiểm tra `endWord in words` trước tiên**. BFS một chiều tự động ra 0 khi `endWord` không có trong từ điển (vì không bao giờ sinh ra nó), nhưng ở đây ta *xuất phát* từ `endWord`, nên phải tự chặn.
- **Kiểm tra `nxt in back` trước `nxt in words`**: các từ trong `back` đã bị xóa khỏi `words`, nếu chỉ tra `words` thì hai sóng "đi xuyên qua nhau" mà không biết.

In [21]:
def ladder_bidirectional(beginWord: str, endWord: str, wordList: List[str]) -> int:
    words = set(wordList)
    if endWord not in words:
        return 0
    words.discard(beginWord)
    words.discard(endWord)
    front, back = {beginWord}, {endWord}
    length = 1                                   # = d_front + d_back + 1
    while front and back:
        if len(front) > len(back):
            front, back = back, front            # luôn mở rộng phía nhỏ hơn
        next_front = set()
        for word in front:
            for i in range(len(word)):
                prefix, suffix = word[:i], word[i + 1:]
                for c in string.ascii_lowercase:
                    nxt = prefix + c + suffix
                    if nxt in back:
                        return length + 1        # hai sóng gặp nhau
                    if nxt in words:
                        words.remove(nxt)
                        next_front.add(nxt)
        front = next_front
        length += 1
    return 0


print(ladder_bidirectional(*EX1))   # 5
print(ladder_bidirectional(*EX2))   # 0
print(ladder_bidirectional("hit", "hot", ["hot"]))   # 2: gặp ngay vòng đầu

5
0
2


### 9.3 Trace BFS hai chiều với `hit → cog`

Ký hiệu: **B** = phía xuất phát từ `beginWord`, **E** = phía xuất phát từ `endWord`. Ban đầu `words = {dog, dot, log, lot, hot}` (đã bỏ `cog`; `hit` vốn không có).

| Vòng | `length` đầu vòng | Mở rộng phía | `front` (đang mở rộng) | `back` | Biến thể quan trọng | Kết quả vòng |
|---:|---:|---|---|---|---|---|
| 1 | 1 | B (1 vs 1, không đổi) | `{hit}` | `{cog}` | `hot` ∈ words | `front = {hot}`, `length = 2` |
| 2 | 2 | B (1 vs 1, không đổi) | `{hot}` | `{cog}` | `dot`, `lot` ∈ words | `front = {dot, lot}`, `length = 3` |
| 3 | 3 | **đổi sang E** (2 > 1) | `{cog}` | `{dot, lot}` | `dog`, `log` ∈ words | `front = {dog, log}`, `length = 4` |
| 4 | 4 | E (2 vs 2, không đổi) | `{dog, log}` | `{dot, lot}` | `dog → dot` ∈ **back**! | **return 4 + 1 = 5** |

Đọc vòng 4 và kiểm tra bất biến `length = d_front + d_back + 1`: phía B đã đi **2** bước (`hit → hot → dot`), phía E đã đi **1** bước (`cog → dog`), nên `length = 2 + 1 + 1 = 4`. Khi `dog` (phía E) chạm `dot` (phía B), đường hoàn chỉnh `hit → hot → dot → dog → cog` có `length + 1 = 5` từ. ✅

So với Cách 3 (thăm cả 6 từ, lấy ra 7 lần), bidirectional chỉ mở rộng `hit, hot, cog` và dừng giữa chừng khi đang mở rộng `dog`. Với ví dụ nhỏ khác biệt chưa lớn; mục 9.4 đo trên từ điển lớn.

Hàm trace dưới duyệt `sorted(front)` để output ổn định (thứ tự duyệt `set` có thể khác nhau giữa các lần chạy; kết quả độ dài thì không đổi).

In [22]:
def trace_bidirectional(beginWord, endWord, wordList):
    words = set(wordList)
    if endWord not in words:
        print("endWord không có trong từ điển -> 0"); return 0
    words.discard(beginWord); words.discard(endWord)
    front, back = {beginWord}, {endWord}
    side_front, side_back = "B", "E"
    length, rnd = 1, 0
    print("words ban đầu:", sorted(words))
    while front and back:
        rnd += 1
        if len(front) > len(back):
            front, back = back, front
            side_front, side_back = side_back, side_front
        print(f"Vòng {rnd}: length={length}, mở rộng {side_front} front={sorted(front)}, back={sorted(back)}")
        next_front = set()
        for word in sorted(front):
            for i in range(len(word)):
                for c in string.ascii_lowercase:
                    nxt = word[:i] + c + word[i + 1:]
                    if nxt in back:
                        print(f"   {word} -> {nxt} nằm trong back: GẶP NHAU -> return {length + 1}")
                        return length + 1
                    if nxt in words:
                        words.remove(nxt); next_front.add(nxt)
                        print(f"   {word} -> {nxt} (mới)")
        front = next_front
        length += 1
    print("Một phía rỗng -> 0"); return 0


trace_bidirectional(*EX1)

words ban đầu: ['dog', 'dot', 'hot', 'log', 'lot']
Vòng 1: length=1, mở rộng B front=['hit'], back=['cog']
   hit -> hot (mới)
Vòng 2: length=2, mở rộng B front=['hot'], back=['cog']
   hot -> dot (mới)
   hot -> lot (mới)
Vòng 3: length=3, mở rộng E front=['cog'], back=['dot', 'lot']
   cog -> dog (mới)
   cog -> log (mới)
Vòng 4: length=4, mở rộng E front=['dog', 'log'], back=['dot', 'lot']
   dog -> dot nằm trong back: GẶP NHAU -> return 5


5

### 9.4 Đo "khối lượng công việc" trên từ điển lớn

Để so công bằng, ta không đo giây (phụ thuộc máy) mà đếm hai đại lượng **cố định**:

- `expanded`: số từ được "mở rộng" (sinh biến thể).
- `candidates`: tổng số chuỗi ứng viên đã sinh và tra `set`.

Từ điển: 2500 từ ngẫu nhiên dài 5 chỉ dùng chữ `a..f` (`random.seed` cố định để ai chạy cũng ra cùng kết quả), đi từ `"aaaaa"` tới `"fffff"` (khác nhau cả 5 vị trí → đáp án ít nhất 6 từ).

In [23]:
def work_bfs_26(beginWord, endWord, wordList):
    word_set = set(wordList); word_set.discard(beginWord)
    if endWord not in word_set: return 0, 0, 0
    q = deque([(beginWord, 1)]); expanded = candidates = 0
    while q:
        word, level = q.popleft()
        if word == endWord: return level, expanded, candidates
        expanded += 1
        for i in range(len(word)):
            for c in string.ascii_lowercase:
                if c == word[i]: continue
                candidates += 1
                nxt = word[:i] + c + word[i + 1:]
                if nxt in word_set: word_set.remove(nxt); q.append((nxt, level + 1))
    return 0, expanded, candidates


def work_bidirectional(beginWord, endWord, wordList):
    words = set(wordList)
    if endWord not in words: return 0, 0, 0
    words.discard(beginWord); words.discard(endWord)
    front, back, length = {beginWord}, {endWord}, 1; expanded = candidates = 0
    while front and back:
        if len(front) > len(back): front, back = back, front
        nf = set()
        for word in sorted(front):                       # sorted: đếm ổn định giữa các lần chạy
            expanded += 1
            for i in range(len(word)):
                for c in string.ascii_lowercase:
                    if c == word[i]: continue
                    candidates += 1
                    nxt = word[:i] + c + word[i + 1:]
                    if nxt in back: return length + 1, expanded, candidates
                    if nxt in words: words.remove(nxt); nf.add(nxt)
        front, length = nf, length + 1
    return 0, expanded, candidates


random.seed(127)
universe = ["".join(t) for t in itertools.product("abcdef", repeat=5)]
big_words = random.sample(universe, 2500) + ["fffff"]
for name, f in [("BFS một chiều (Cách 3)", work_bfs_26), ("BFS hai chiều (Cách 5)", work_bidirectional)]:
    ans, ex, ca = f("aaaaa", "fffff", big_words)
    print(f"{name}: đáp án={ans}, expanded={ex:,}, candidates={ca:,}")

BFS một chiều (Cách 3): đáp án=6, expanded=1,171, candidates=146,375
BFS hai chiều (Cách 5): đáp án=6, expanded=65, candidates=8,080


Hai cách cho **cùng đáp án 6**, nhưng BFS một chiều phải mở rộng **1 171** từ (sinh 146 375 ứng viên) trước khi chạm `"fffff"`, còn BFS hai chiều chỉ mở rộng **65** từ (8 080 ứng viên) — ít hơn khoảng **18 lần**. Lý do: BFS một chiều phải quét hết các tầng 1..5 quanh `"aaaaa"` (càng xa càng đông), còn hai sóng gặp nhau ở giữa nên mỗi bên chỉ cần đi khoảng nửa đường, nơi các tầng còn "mỏng".

**Độ phức tạp Cách 5**:

- **Thời gian trường hợp xấu nhất** vẫn là `O(26 · N · L²)` như Cách 3 (mỗi từ được mở rộng tối đa một lần), nhưng **thực tế** nhanh hơn nhiều nhờ chỉ thăm cỡ `2·b^(d/2)` thay vì `b^d` từ, cộng với mẹo luôn mở rộng phía nhỏ hơn.
- **Bộ nhớ `O(N · L)`**: `words`, `front`, `back`.

Có thể kết hợp bidirectional với wildcard bucket (Cách 4) để bỏ hệ số 26, nhưng code dài hơn; với `L ≤ 10` bản trên đã rất nhanh trên LeetCode.

## 10. Code nộp LeetCode

Chuyển hàm `ladder_bidirectional` vào đúng khung `class Solution`:

- Tên method `ladderLength`, có `self`, giữ type hint `List[str]` (LeetCode đã `from typing import List` sẵn; trong notebook ta import ở cell đầu).
- Bỏ mọi `print`.
- Dùng chuỗi chữ cái `"abcdefghijklmnopqrstuvwxyz"` hoặc `string.ascii_lowercase` (LeetCode cho dùng module `string`).

### 10.1 Bản chính — BFS hai chiều

In [24]:
class Solution:
    def ladderLength(self, beginWord: str, endWord: str, wordList: List[str]) -> int:
        words = set(wordList)
        if endWord not in words:
            return 0
        words.discard(beginWord)
        words.discard(endWord)
        front, back = {beginWord}, {endWord}
        length = 1
        letters = "abcdefghijklmnopqrstuvwxyz"
        while front and back:
            if len(front) > len(back):
                front, back = back, front
            next_front = set()
            for word in front:
                for i in range(len(word)):
                    prefix, suffix = word[:i], word[i + 1:]
                    for c in letters:
                        nxt = prefix + c + suffix
                        if nxt in back:
                            return length + 1
                        if nxt in words:
                            words.remove(nxt)
                            next_front.add(nxt)
            front = next_front
            length += 1
        return 0


sol = Solution()
assert sol.ladderLength("hit", "cog", ["hot", "dot", "dog", "lot", "log", "cog"]) == 5
assert sol.ladderLength("hit", "cog", ["hot", "dot", "dog", "lot", "log"]) == 0
print("Example 1, 2: PASS")

Example 1, 2: PASS


### 10.2 Bản phụ — BFS một chiều (dễ nhớ nhất khi phỏng vấn)

Nếu bạn sợ nhầm bất biến `length` của bản hai chiều, bản BFS một chiều dưới đây cũng **Accepted** trên LeetCode và rất dễ tự viết lại.

In [25]:
class SolutionSimpleBFS:
    def ladderLength(self, beginWord: str, endWord: str, wordList: List[str]) -> int:
        word_set = set(wordList)
        if endWord not in word_set:
            return 0
        word_set.discard(beginWord)
        q = deque([(beginWord, 1)])
        while q:
            word, level = q.popleft()
            if word == endWord:
                return level
            for i in range(len(word)):
                for c in "abcdefghijklmnopqrstuvwxyz":
                    nxt = word[:i] + c + word[i + 1:]
                    if nxt in word_set:
                        word_set.remove(nxt)
                        q.append((nxt, level + 1))
        return 0


assert SolutionSimpleBFS().ladderLength(*EX1) == 5
assert SolutionSimpleBFS().ladderLength(*EX2) == 0
print("SolutionSimpleBFS: PASS")

SolutionSimpleBFS: PASS


(Ở bản một chiều này không cần `if c == word[i]: continue`: khi `c == word[i]` thì `nxt == word`, mà `word` đã bị xóa khỏi `word_set` từ lúc vào queue — `beginWord` thì được `discard` ngay đầu — nên phép tra tự động ra `False`.)

### 10.3 Edge case cần test

| Trường hợp | Input | Mong đợi | Vì sao |
|---|---|---:|---|
| `beginWord` kề thẳng `endWord` | `"hit","hot",["hot"]` | 2 | đường 2 từ |
| `endWord` không có | Example 2 | 0 | điều kiện `sk ∈ wordList` |
| Từ 1 ký tự | `"a","c",["a","b","c"]` | 2 | `a` kề `c` trực tiếp (mọi từ 1 chữ đều kề nhau) |
| `beginWord` nằm trong `wordList` | `"hot","dog",["hot","dog","dot"]` | 3 | `hot-dot-dog`; không được đếm `beginWord` hai lần |
| Có `endWord` nhưng không nối được | `"hot","dog",["hot","dog"]` | 0 | `hot`, `dog` khác 2 chữ, không có từ trung gian |
| Phải đổi cả 3 vị trí | `"abc","xyz",[...]` (cell dưới) | 4 | hai đường 4 từ: `abc→xbc→xyc→xyz` và `abc→abz→ayz→xyz` |
| Thứ tự `wordList` "xấu" cho DFS | `reordered` | 5 | BFS không phụ thuộc thứ tự |

In [26]:
methods = {
    "explicit_graph": ladder_explicit_graph,
    "bfs_26": ladder_bfs_26,
    "wildcard": ladder_wildcard,
    "bidirectional": ladder_bidirectional,
    "Solution": Solution().ladderLength,
    "SolutionSimpleBFS": SolutionSimpleBFS().ladderLength,
}

edge_cases = [
    (("hit", "hot", ["hot"]), 2),
    (EX2, 0),
    (("a", "c", ["a", "b", "c"]), 2),
    (("hot", "dog", ["hot", "dog", "dot"]), 3),
    (("hot", "dog", ["hot", "dog"]), 0),
    (("abc", "xyz", ["xbc", "xyc", "xyz", "abz", "ayz"]), 4),
    (("hit", "cog", reordered), 5),
    (EX1, 5),
]

for (args, expected) in edge_cases:
    for name, f in methods.items():
        got = f(*args)
        assert got == expected, (name, args, got, expected)
    print(f"{str(args)[:58]:60s} -> {expected}  PASS")

('hit', 'hot', ['hot'])                                      -> 2  PASS
('hit', 'cog', ['hot', 'dot', 'dog', 'lot', 'log'])          -> 0  PASS
('a', 'c', ['a', 'b', 'c'])                                  -> 2  PASS
('hot', 'dog', ['hot', 'dog', 'dot'])                        -> 3  PASS
('hot', 'dog', ['hot', 'dog'])                               -> 0  PASS
('abc', 'xyz', ['xbc', 'xyc', 'xyz', 'abz', 'ayz'])          -> 4  PASS
('hit', 'cog', ['hot', 'lot', 'dot', 'dog', 'log', 'cog'])   -> 5  PASS
('hit', 'cog', ['hot', 'dot', 'dog', 'lot', 'log', 'cog'])   -> 5  PASS


### 10.4 Test đối chiếu chéo trên input ngẫu nhiên

Ta sinh nhiều bộ input nhỏ ngẫu nhiên (`random.seed` cố định) và yêu cầu **mọi cách** cho cùng kết quả:

- Bộ **nhỏ** (từ dài 1–3, chữ `a..c`, tối đa 8 từ): so thêm với DFS vét cạn (Cách 1) làm "trọng tài" — chậm nhưng chắc chắn đúng vì thử hết mọi đường.
- Bộ **vừa** (từ dài 3–4, chữ `a..e`, tối đa 60 từ): DFS quá chậm nên chỉ so Cách 2–5 với nhau.

In [27]:
def random_case(rng, max_len, alphabet, max_words):
    L = rng.randint(1, max_len)
    universe = ["".join(t) for t in itertools.product(alphabet, repeat=L)]
    wl = rng.sample(universe, rng.randint(1, min(max_words, len(universe))))
    begin = rng.choice(universe)
    end = rng.choice(wl) if rng.random() < 0.85 else rng.choice(universe)
    if begin == end:
        return None
    return begin, end, wl


rng = random.Random(2024)
small = medium = 0
while small < 400:
    case = random_case(rng, 3, "abc", 8)
    if case is None: continue
    ref, _ = ladder_dfs_all_paths(*case)
    for name, f in methods.items():
        assert f(*case) == ref, (name, case, f(*case), ref)
    small += 1
while medium < 300:
    case = random_case(rng, 4, "abcde", 60)
    if case is None: continue
    ref = ladder_bfs_26(*case)
    for name, f in methods.items():
        assert f(*case) == ref, (name, case)
    medium += 1
print(f"Đối chiếu {small} bộ nhỏ (có DFS trọng tài) và {medium} bộ vừa: tất cả khớp!")

Đối chiếu 400 bộ nhỏ (có DFS trọng tài) và 300 bộ vừa: tất cả khớp!


### 10.5 Thử với kích thước gần giới hạn đề

Dựng một từ điển `N = 5000` từ, `L = 10`:

- 4989 từ ngẫu nhiên chỉ gồm chữ `a, b, c` (seed cố định, có sắp xếp trước khi chọn để ai chạy cũng được cùng từ điển);
- một chuỗi 10 từ "nối" chắc chắn `aaaaaaaaaa → caaaaaaaaa → ccaaaaaaaa → ... → cccccccccc` (nên đáp án (a) là 11 — không thể ít hơn vì hai đầu khác nhau cả 10 vị trí);
- một từ **cô lập** `zzzzzzzzzz`.

Hai truy vấn từ `beginWord = "aaaaaaaaaa"`:

- **(a)** tới `"cccccccccc"` — có đường.
- **(b)** tới `"zzzzzzzzzz"` — có trong từ điển nhưng không nối được. BFS một chiều phải quét **hết** thành phần liên thông của `beginWord` mới dám kết luận `0`; BFS hai chiều thấy ngay phía `zzzzzzzzzz` không có hàng xóm nào → mặt sóng rỗng → dừng gần như tức thì.

Cách 2 chỉ chạy một lần cho (a) vì nó mất **vài giây** (12,5 triệu cặp). Thời gian cụ thể tùy máy; điều cần thấy là khoảng cách giữa Cách 2 và Cách 3–5 lên tới hàng trăm, hàng nghìn lần.

Một chi tiết thú vị: Cách 4 phải dựng bucket cho **cả** 5000 từ ngay từ đầu, nên ở truy vấn (a) — nơi BFS chỉ cần thăm vài trăm từ — nó có thể chậm hơn Cách 3. Big-O tốt hơn không có nghĩa là luôn nhanh hơn trên mọi input; chi phí tiền xử lý cũng đáng kể.

In [28]:
rng = random.Random(5000)
L = 10
begin_w = "a" * L
chain, cur = [], list(begin_w)
for i in range(L):                       # đổi lần lượt từng vị trí a -> c
    cur[i] = "c"; chain.append("".join(cur))
pool = {"".join(rng.choice("abc") for _ in range(L)) for _ in range(6000)} - set(chain) - {begin_w}
big_list = sorted(pool)[: 4999 - len(chain)] + chain + ["z" * L]
rng.shuffle(big_list)
print("N =", len(big_list), "| L =", L)

for end_w in ["c" * L, "z" * L]:
    print(f"Truy vấn tới {end_w}:")
    names = ["explicit_graph", "bfs_26", "wildcard", "bidirectional"] if end_w[0] == "c" else ["bfs_26", "wildcard", "bidirectional"]
    for name in names:
        t0 = time.perf_counter(); ans = methods[name](begin_w, end_w, big_list); t = time.perf_counter() - t0
        print(f"  {name:14s}: đáp án = {ans:2d}, thời gian = {t:.3f}s")

_, ex1, _ = work_bfs_26(begin_w, "z" * L, big_list)
_, ex2, _ = work_bidirectional(begin_w, "z" * L, big_list)
print(f"Truy vấn (b): BFS một chiều mở rộng {ex1:,} từ, BFS hai chiều mở rộng {ex2} từ")

N = 5000 | L = 10
Truy vấn tới cccccccccc:


  explicit_graph: đáp án = 11, thời gian = 6.801s
  bfs_26        : đáp án = 11, thời gian = 0.010s
  wildcard      : đáp án = 11, thời gian = 0.038s
  bidirectional : đáp án = 11, thời gian = 0.001s
Truy vấn tới zzzzzzzzzz:
  bfs_26        : đáp án =  0, thời gian = 0.113s
  wildcard      : đáp án =  0, thời gian = 0.037s
  bidirectional : đáp án =  0, thời gian = 0.000s


Truy vấn (b): BFS một chiều mở rộng 3,638 từ, BFS hai chiều mở rộng 2 từ


## 11. Vì sao lời giải đúng?

### 11.1 BFS một chiều (Cách 2, 3, 4)

**Bất biến (invariant)**: các từ được lấy ra khỏi queue theo thứ tự tầng **không giảm**: hết tầng 1, rồi tầng 2, rồi tầng 3... và khi một từ được đưa vào queue với tầng `level + 1`, đó là **số từ ít nhất** của một dãy hợp lệ từ `beginWord` tới nó.

Lý do bằng lời:

1. Queue FIFO, mỗi từ tầng `d` chỉ đẩy vào các từ tầng `d+1` (ở cuối queue) → mọi từ tầng `d` ra trước mọi từ tầng `d+1`.
2. Giả sử từ `v` có khoảng cách thật là `k`. Trên đường ngắn nhất tới `v`, từ ngay trước nó là `u` có khoảng cách `k−1`; theo quy nạp `u` được lấy ra ở tầng `k−1`, và khi đó nếu `v` chưa thăm thì `v` được gán tầng `k`. Nếu `v` đã thăm thì nó được gán **sớm hơn**, tức là từ một tầng `≤ k−1` → tầng của `v` `≤ k`; mà không thể `< k` vì `k` là khoảng cách thật. Vậy tầng của `v` đúng bằng `k`.
3. Do đó xóa từ khỏi `word_set` (đánh dấu đã thăm) ngay lần đầu gặp **không làm mất** đường ngắn nhất nào.
4. Chỉ những từ trong `wordList` được vào queue → mọi dãy tìm được đều hợp lệ. Nếu `endWord` không có trong từ điển, nó không bao giờ vào queue → trả `0` đúng như Example 2 (dù ta vẫn kiểm tra sớm cho nhanh).

### 11.2 BFS hai chiều (Cách 5)

Gọi `D` là số **cạnh** của đường ngắn nhất. **Bất biến** đầu mỗi vòng: `front`, `back` đúng là tầng `d_front`, `d_back` của mỗi phía, `length = d_front + d_back + 1`, và **`D ≥ d_front + d_back + 1`** (chưa có đường nào ngắn hơn hoặc bằng `d_front + d_back` cạnh, nếu không ta đã phát hiện ở vòng trước).

Khi mở rộng `front`:

- Nếu một biến thể `nxt` của `word ∈ front` nằm trong `back`: có đường `d_front + 1 + d_back` cạnh, kết hợp với bất biến → đó là **ngắn nhất**, số từ = `d_front + d_back + 2 = length + 1`.
- Nếu không có cặp nào chạm nhau: một đường dài đúng `d_front + d_back + 1` cạnh sẽ phải có một cạnh nối một từ tầng `d_front` (phía front) với một từ tầng `d_back` (phía back) — ta vừa kiểm tra hết mà không thấy → `D ≥ d_front + d_back + 2`. Sau `length += 1` bất biến được giữ.

Vì sao xóa khỏi `words` không làm mất lời giải: nếu `nxt` đã bị phía **kia** nhận ở một tầng **cũ hơn** `d_back`, thì đã có đường ngắn hơn `D`, mâu thuẫn bất biến. Nên trường hợp duy nhất quan trọng là `nxt ∈ back` — chính là điều ta kiểm tra trước.

### 11.3 DFS (Cách 1) đúng nhưng chậm

DFS có quay lui liệt kê **mọi** đường đơn, nên chắc chắn gặp đường ngắn nhất; giữ `min` → đúng. Còn DFS có `visited` vĩnh viễn thì **sai** (phản ví dụ `reordered` cho 7 thay vì 5).

## 12. Bảng so sánh các cách

Ký hiệu: `N = len(wordList) ≤ 5000`, `L = len(beginWord) ≤ 10`, `E` = số cạnh của đồ thị (có thể tới `O(N²)`), `b` = hệ số rẽ nhánh, `d` = số bước của đáp án.

| Cách | Ý tưởng | Thời gian | Bộ nhớ | Nhận xét |
|---|---|---|---|---|
| 1. DFS thử mọi đường | Quay lui liệt kê mọi đường đơn, giữ min | Hàm mũ, tệ nhất `O(N!·N·L)` | `O(N)` | Sai nếu dùng `visited` vĩnh viễn; TLE ngay cả với `N ≈ 30` |
| 2. Đồ thị tường minh + BFS | So mọi cặp để dựng danh sách kề | `O(N²·L + E)` | `O(N + E)` | Đúng, nhưng 125 triệu phép so khi `N=5000` → dễ TLE |
| 3. BFS thay từng chữ `a–z` | Sinh `25·L` biến thể, tra `set` | `O(26·N·L²)` | `O(N·L)` | Chuẩn mực, dễ viết; tuyến tính theo `N` |
| 4. BFS + wildcard bucket | Nhóm từ theo khuôn `h*t` | `O(N·L²)` | `O(N·L²)` | Bỏ hệ số 26; nhớ xóa bucket sau khi dùng |
| 5. BFS hai chiều | Hai sóng từ hai đầu, luôn mở rộng phía nhỏ | Xấu nhất `O(26·N·L²)`, thực tế cỡ `2·b^(d/2)` thay vì `b^d` từ | `O(N·L)` | Nhanh nhất thực tế; bản nộp chính |

**Khuyên dùng**: học thuộc Cách 3 như "phản xạ" BFS; nộp Cách 5 khi muốn nhanh nhất; hiểu Cách 4 vì kỹ thuật bucket xuất hiện lại ở #126 và nhiều bài nhóm chuỗi.

## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đếm số bước thay vì số từ (off-by-one)

```python
q = deque([(beginWord, 0)])     # SAI: bắt đầu từ 0
```

→ Example 1 trả `4`. **Sửa**: tầng xuất phát là `1` vì `beginWord` đã là một từ trong dãy.

### Lỗi 2: Đánh dấu `visited` lúc lấy ra thay vì lúc đưa vào

Kết quả vẫn đúng nhưng một từ có thể vào queue rất nhiều lần (mục 3.7 đã đếm: 436 lần `append` thay vì 30 trên đồ thị đầy đủ 30 đỉnh) → TLE trên từ điển "dày".

### Lỗi 3: Dùng `wordList` (list) để tra `in`

`nxt in wordList` tốn `O(N)` mỗi lần → Cách 3 thành `O(26·N²·L²)`. **Sửa**: `word_set = set(wordList)`.

### Lỗi 4: Dùng `list.pop(0)` làm queue

Mỗi lần `O(N)`. **Sửa**: `collections.deque` + `popleft()`.

### Lỗi 5: DFS với `visited` vĩnh viễn

Ra 7 thay vì 5 (mục 4). **Sửa**: dùng BFS.

Cell dưới chạy lỗi 1 để thấy tận mắt.

In [29]:
def ladder_wrong_steps(beginWord, endWord, wordList):
    word_set = set(wordList)
    if endWord not in word_set:
        return 0
    q = deque([(beginWord, 0)])              # SAI: đếm bước
    while q:
        word, level = q.popleft()
        if word == endWord:
            return level
        for i in range(len(word)):
            for c in string.ascii_lowercase:
                nxt = word[:i] + c + word[i + 1:]
                if nxt in word_set:
                    word_set.remove(nxt); q.append((nxt, level + 1))
    return 0


print("Lỗi 1 — đếm bước:", ladder_wrong_steps(*EX1), "| đúng:", ladder_bfs_26(*EX1))

Lỗi 1 — đếm bước: 4 | đúng: 5


### Lỗi 6 (bidirectional): quên kiểm tra `endWord in wordList`

Phía E xuất phát thẳng từ `endWord` nên dù `"cog"` không có trong từ điển, hai sóng vẫn gặp nhau → trả `5` cho Example 2 (đúng phải là `0`).

### Lỗi 7 (bidirectional): tra `words` trước, rồi mới tra `back` bên trong

```python
if nxt in words:          # các từ trong back ĐÃ bị xóa khỏi words
    if nxt in back:       # -> không bao giờ True
        return length + 1
```

Hai sóng đi xuyên qua nhau, cạn từ → trả `0` cho Example 1.

### Lỗi 8 (bidirectional): `return length` thay vì `length + 1`, hoặc tăng `length` sai chỗ

Luôn tự kiểm tra bằng ca nhỏ nhất `"hit","hot",["hot"]` → phải ra `2`.

In [30]:
def bidir_no_end_check(beginWord, endWord, wordList):
    words = set(wordList)                     # SAI: thiếu "if endWord not in words: return 0"
    words.discard(beginWord); words.discard(endWord)
    front, back, length = {beginWord}, {endWord}, 1
    while front and back:
        if len(front) > len(back): front, back = back, front
        nf = set()
        for w in front:
            for i in range(len(w)):
                for c in string.ascii_lowercase:
                    nxt = w[:i] + c + w[i + 1:]
                    if nxt in back: return length + 1
                    if nxt in words: words.remove(nxt); nf.add(nxt)
        front, length = nf, length + 1
    return 0


def bidir_wrong_order(beginWord, endWord, wordList):
    words = set(wordList)
    if endWord not in words: return 0
    words.discard(beginWord); words.discard(endWord)
    front, back, length = {beginWord}, {endWord}, 1
    while front and back:
        if len(front) > len(back): front, back = back, front
        nf = set()
        for w in front:
            for i in range(len(w)):
                for c in string.ascii_lowercase:
                    nxt = w[:i] + c + w[i + 1:]
                    if nxt in words:                  # SAI thứ tự
                        if nxt in back: return length + 1
                        words.remove(nxt); nf.add(nxt)
        front, length = nf, length + 1
    return 0


print("Lỗi 6 — Example 2:", bidir_no_end_check(*EX2), "| đúng:", ladder_bidirectional(*EX2))
print("Lỗi 7 — Example 1:", bidir_wrong_order(*EX1), "| đúng:", ladder_bidirectional(*EX1))

Lỗi 6 — Example 2: 5 | đúng: 0
Lỗi 7 — Example 1: 0 | đúng: 5


## 14. Mẹo nhận dạng pattern và bài liên quan

### Dấu hiệu "đây là BFS trên đồ thị ngầm"

- Có một **trạng thái** (từ, mã khóa, chuỗi gen, vị trí trên lưới...) và một **phép biến đổi nhỏ** (đổi 1 chữ, xoay 1 bánh số, đi 1 ô).
- Hỏi **số bước ít nhất** / **đường ngắn nhất** / "có tới được không".
- Mỗi bước có **chi phí như nhau** (không trọng số). Nếu chi phí khác nhau → nghĩ tới Dijkstra.
- Không gian trạng thái lớn nhưng mỗi trạng thái có ít "hàng xóm" sinh được bằng quy tắc → **đừng dựng đồ thị trước**, sinh hàng xóm tại chỗ.
- Biết cả điểm đầu và điểm cuối, và phép biến đổi đảo ngược được → cân nhắc **BFS hai chiều**.

### Câu thần chú

> *"Ngắn nhất + không trọng số → BFS. Hàng xóm sinh theo quy tắc → đồ thị ngầm. Đánh dấu lúc đưa vào queue."*

### Bài liên quan

| Bài | Liên hệ |
|---|---|
| #126 Word Ladder II (Hard) | Liệt kê **mọi** đường ngắn nhất: BFS theo tầng lưu cha + DFS/backtrack dựng đường |
| #433 Minimum Genetic Mutation (Medium) | Giống hệt, bảng chữ `A C G T`, đếm **bước** (không phải số từ), trả `-1` khi không được |
| #752 Open the Lock (Medium) | Trạng thái `"0000"`, mỗi bước xoay 1 bánh ±1, có `deadends` — BFS / BFS hai chiều |
| #841 Keys and Rooms, #1971 Find if Path Exists | BFS/DFS cơ bản "có tới được không" |
| #1091 Shortest Path in Binary Matrix | BFS trên lưới, 8 hướng |
| #815 Bus Routes (Hard) | BFS trên đồ thị ngầm mà "đỉnh" là tuyến xe |
| #49 Group Anagrams | Kỹ thuật "nhóm theo khóa chuẩn hóa" giống wildcard bucket |

## 15. Bài tập vận dụng

Làm theo thứ tự, mỗi bài khó hơn một chút. **Đừng xem đáp án** (mục 17) trước khi tự thử. Các cell TODO chỉ định nghĩa hàm, các `assert` đang bị comment — làm xong thì bỏ comment để tự kiểm tra.

### Bài 1 — Tính tay bảng tầng BFS

Cho:

```text
beginWord = "red"
endWord   = "tax"
wordList  = ["ted","tex","red","tax","tad","den","rex","pee"]
```

1. Vẽ đồ thị: từ nào kề từ nào?
2. Lập bảng tầng BFS từ `"red"` (tầng 1). `"tax"` ở tầng mấy? → đáp án `ladderLength`.
3. Có bao nhiêu dãy ngắn nhất khác nhau? Liệt kê chúng.
4. Từ nào **không bao giờ** được thăm? Vì sao?

Ghi đáp án của bạn vào cell dưới.

In [31]:
# TODO: điền đáp án tính tay của bạn
my_ladder_length_bai1 = None          # số nguyên
my_unreachable_bai1 = None            # set các từ không bao giờ được thăm

# Tự kiểm tra (bỏ comment sau khi điền):
# assert my_ladder_length_bai1 == ladder_bfs_26("red", "tax", ["ted","tex","red","tax","tad","den","rex","pee"])

### Bài 2 — Điền TODO: sinh hàng xóm

Viết `neighbors_in_set(word, word_set)` trả về **list** các từ trong `word_set` khác `word` đúng 1 chữ, theo thứ tự: vị trí `i` tăng dần, trong mỗi vị trí chữ `a → z`.

```python
neighbors_in_set("hot", {"hot","dot","dog","lot","log","cog"})  ->  ["dot", "lot"]
neighbors_in_set("dog", {"hot","dot","dog","lot","log","cog"})  ->  ["cog", "log", "dot"]
```

In [32]:
def neighbors_in_set(word: str, word_set: set) -> list:
    result = []
    # TODO 1: duyệt i từ 0 tới len(word) - 1
    # TODO 2: duyệt c trong string.ascii_lowercase, bỏ qua c == word[i]
    # TODO 3: ghép nxt = word[:i] + c + word[i+1:]
    # TODO 4: nếu nxt có trong word_set thì thêm vào result
    pass

# Tự kiểm tra:
# S = {"hot","dot","dog","lot","log","cog"}
# assert neighbors_in_set("hot", S) == ["dot", "lot"]
# assert neighbors_in_set("dog", S) == ["cog", "log", "dot"]
# assert neighbors_in_set("xyz", S) == []

### Bài 3 — Tự viết lại BFS một chiều

Không nhìn mục 7, tự viết `ladder_bfs_exercise(beginWord, endWord, wordList)` bằng `deque`, `set`, và đánh dấu đã thăm **lúc đưa vào queue**.

In [33]:
def ladder_bfs_exercise(beginWord: str, endWord: str, wordList: List[str]) -> int:
    # TODO 1: word_set = set(wordList); nếu endWord không có -> return 0
    # TODO 2: queue bắt đầu với (beginWord, 1)  -- nhớ: đếm SỐ TỪ
    # TODO 3: vòng while: popleft, nếu là endWord -> return level
    # TODO 4: sinh biến thể, cái nào trong word_set -> xóa khỏi set, append (nxt, level + 1)
    # TODO 5: hết queue -> return 0
    raise NotImplementedError

# Tự kiểm tra:
# assert ladder_bfs_exercise(*EX1) == 5
# assert ladder_bfs_exercise(*EX2) == 0
# assert ladder_bfs_exercise("a", "c", ["a", "b", "c"]) == 2
# assert ladder_bfs_exercise("hot", "dog", ["hot", "dog"]) == 0

### Bài 4 — Biến thể: trả về **một** dãy ngắn nhất

Viết `shortest_ladder_path(beginWord, endWord, wordList)` trả về **list các từ** của một dãy ngắn nhất (bất kỳ dãy nào nếu có nhiều), hoặc `[]` nếu không có.

Ví dụ: Example 1 có thể trả `["hit","hot","dot","dog","cog"]` hoặc `["hit","hot","lot","log","cog"]`.

(Đây là bước đệm cho #126 Word Ladder II — bài đó cần **mọi** dãy.)

In [34]:
def shortest_ladder_path(beginWord: str, endWord: str, wordList: List[str]) -> List[str]:
    # TODO 1: BFS như bài 3, nhưng thêm dict parent: parent[nxt] = word khi đưa nxt vào queue
    # TODO 2: khi lấy ra endWord, đi ngược parent từ endWord về beginWord
    # TODO 3: đảo ngược list rồi trả về
    raise NotImplementedError

# Tự kiểm tra:
# p = shortest_ladder_path(*EX1)
# assert len(p) == 5 and is_valid_ladder(p, *EX1)
# assert shortest_ladder_path(*EX2) == []

### Bài 5 — Chuyển giao: Minimum Genetic Mutation (LeetCode 433)

Một chuỗi gen dài 8 ký tự, mỗi ký tự thuộc `'A', 'C', 'G', 'T'`. Một **đột biến** đổi đúng 1 ký tự. Mỗi gen trung gian (và gen đích) phải nằm trong `bank`. Trả về **số đột biến ít nhất** (số **bước**, không phải số chuỗi!) từ `startGene` tới `endGene`, hoặc `-1` nếu không thể. `startGene` coi như hợp lệ dù không có trong `bank`.

```python
min_mutation("AACCGGTT", "AACCGGTA", ["AACCGGTA"])                          ->  1
min_mutation("AACCGGTT", "AAACGGTA", ["AACCGGTA","AACCGCTA","AAACGGTA"])    ->  2
```

Chú ý ba điểm khác Word Ladder: bảng chữ 4 ký tự, đếm bước, trả `-1`.

In [35]:
def min_mutation(startGene: str, endGene: str, bank: List[str]) -> int:
    # TODO: BFS giống Word Ladder, chỉ đổi bảng chữ thành "ACGT",
    #       tầng xuất phát là 0 (đếm bước), không tới được thì return -1
    raise NotImplementedError

# Tự kiểm tra:
# assert min_mutation("AACCGGTT", "AACCGGTA", ["AACCGGTA"]) == 1
# assert min_mutation("AACCGGTT", "AAACGGTA", ["AACCGGTA", "AACCGCTA", "AAACGGTA"]) == 2
# assert min_mutation("AACCGGTT", "AACCGGTA", []) == -1

### Bài 6 — Mini challenge: Open the Lock (LeetCode 752)

Ổ khóa có 4 bánh xe, mỗi bánh hiện một chữ số `'0'..'9'`, xoay vòng được (`'9'` xoay lên thành `'0'`, `'0'` xoay xuống thành `'9'`). Ban đầu khóa ở `"0000"`. Mỗi bước xoay **một** bánh lên hoặc xuống 1 nấc. `deadends` là danh sách trạng thái "chết": chạm vào là kẹt, không được đi qua. Trả về số bước ít nhất để tới `target`, hoặc `-1`.

```python
open_lock(["0201","0101","0102","1212","2002"], "0202")                     ->  6
open_lock(["8888"], "0009")                                                 ->  1
open_lock(["8887","8889","8878","8898","8788","8988","7888","9888"], "8888") ->  -1
```

Đây là **đồ thị ngầm thuần túy**: không có "từ điển" — mọi chuỗi 4 chữ số đều là đỉnh, trừ `deadends`. Mỗi trạng thái có đúng 8 hàng xóm. Thử viết bằng BFS một chiều trước; nếu muốn thử thách thêm, viết bằng BFS hai chiều.

In [36]:
def open_lock(deadends: List[str], target: str) -> int:
    # TODO 1: dead = set(deadends); nếu "0000" trong dead -> return -1
    # TODO 2: BFS từ ("0000", 0); visited chứa "0000"
    # TODO 3: hàng xóm: với mỗi vị trí i, chữ số d -> (d+1)%10 và (d-1)%10
    # TODO 4: bỏ qua hàng xóm trong dead hoặc đã thăm
    raise NotImplementedError

# Tự kiểm tra:
# assert open_lock(["0201", "0101", "0102", "1212", "2002"], "0202") == 6
# assert open_lock(["8888"], "0009") == 1
# assert open_lock(["8887", "8889", "8878", "8898", "8788", "8988", "7888", "9888"], "8888") == -1
# assert open_lock(["0000"], "8888") == -1

## 16. Gợi ý theo mức

Mở từng mức một; chỉ đọc mức sau nếu mức trước chưa đủ.

**Bài 1**
- Mức 1: so từng cặp từ cùng vị trí để tìm cạnh (giống mục 3.2). `"red"` nằm trong `wordList` nhưng nó chính là điểm xuất phát.
- Mức 2: `red` kề `ted` và `rex`. Tiếp tục lan ra từng tầng, nhớ mỗi từ chỉ thăm một lần.
- Mức 3: `tax` ở tầng 4. Có 3 dãy ngắn nhất. Hai từ không kề bất kỳ từ nào khác.

**Bài 2**
- Mức 1: đây chính là vòng lặp sinh ứng viên ở mục 3.4.
- Mức 2: hai vòng `for` lồng nhau; vòng ngoài là vị trí, vòng trong là chữ cái — thứ tự này quyết định thứ tự output.
- Mức 3: `nxt = word[:i] + c + word[i + 1:]`; `if nxt in word_set: result.append(nxt)`; cuối cùng `return result`.

**Bài 3**
- Mức 1: mẫu BFS ở mục 3.6 + hàm sinh hàng xóm của bài 2.
- Mức 2: `q = deque([(beginWord, 1)])`, `word_set.discard(beginWord)`.
- Mức 3: trong vòng `while q:` → `word, level = q.popleft()`; nếu `word == endWord: return level`; với mỗi `nxt` hợp lệ: `word_set.remove(nxt); q.append((nxt, level + 1))`.

**Bài 4**
- Mức 1: BFS chỉ cho biết khoảng cách; muốn dựng đường phải nhớ "ai đưa mình vào queue".
- Mức 2: `parent = {beginWord: None}` vừa làm visited vừa lưu cha.
- Mức 3: `path = []; cur = endWord; while cur is not None: path.append(cur); cur = parent[cur]`; rồi `return path[::-1]`.

**Bài 5**
- Mức 1: so với `ladder_bfs_26`, chỉ đổi 3 chỗ.
- Mức 2: `for c in "ACGT"`; `q = deque([(startGene, 0)])`; cuối hàm `return -1`.
- Mức 3: kiểm tra `if endGene not in bank_set: return -1` ở đầu cho gọn.

**Bài 6**
- Mức 1: viết hàm `neighbors(state)` sinh 8 trạng thái trước, test riêng nó: `neighbors("0000")` phải chứa `"1000"` và `"9000"`.
- Mức 2: `d = int(state[i])`; `for nd in ((d + 1) % 10, (d - 1) % 10)`; `state[:i] + str(nd) + state[i + 1:]`. Trong Python, `(0 - 1) % 10 == 9`.
- Mức 3: `visited = {"0000"} | dead` — gộp luôn deadends vào visited để chỉ cần một phép kiểm tra. Nhớ xử lý trường hợp `target == "0000"` trả `0`.

## 17. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

Cạnh (kề nhau): `red–ted`, `red–rex`, `ted–tad`, `ted–tex`, `tex–rex`, `tex–tax`, `tad–tax`. Hai từ `den`, `pee` không kề từ nào.

| Tầng | Từ | Đến từ |
|---:|---|---|
| 1 | `red` | xuất phát |
| 2 | `ted`, `rex` | `red` |
| 3 | `tad`, `tex` | `ted` → `tad`, `ted` → `tex` (`rex` → `tex` cũng được nhưng `tex` đã được nhận) |
| 4 | `tax` | `tad` (hoặc `tex`) |

- `ladderLength = 4`.
- 3 dãy ngắn nhất: `red→ted→tad→tax`, `red→ted→tex→tax`, `red→rex→tex→tax`.
- Không bao giờ được thăm: `den` và `pee`. Ví dụ `red`/`den` khác ở **2** vị trí (0 và 2), `ted`/`den` cũng khác 2 vị trí; `pee` khác mọi từ ít nhất 2 chữ. Chúng là các đỉnh **cô lập** (không có cạnh nào), nên BFS không bao giờ chạm tới.

Cell dưới kiểm chứng bằng `trace_bfs_26` và DFS liệt kê mọi đường.

In [37]:
wl_bai1 = ["ted", "tex", "red", "tax", "tad", "den", "rex", "pee"]
trace_bfs_26("red", "tax", wl_bai1)

answer_ladder_length_bai1 = 4
answer_unreachable_bai1 = {"den", "pee"}
adj_bai1 = build_adjacency("red", wl_bai1)
assert answer_ladder_length_bai1 == ladder_bidirectional("red", "tax", wl_bai1)
assert {w for w, nb in adj_bai1.items() if not nb} == answer_unreachable_bai1
shortest = [p for p in all_simple_paths(adj_bai1, "red", "tax") if len(p) == 4]
print("Các dãy ngắn nhất:", [" -> ".join(p) for p in shortest])
assert len(shortest) == 3

Bước 1: lấy red(1) | mới: ['ted', 'rex'] | queue: ted(2) rex(2) | còn: ['den', 'pee', 'tad', 'tax', 'tex']
Bước 2: lấy ted(2) | mới: ['tad', 'tex'] | queue: rex(2) tad(3) tex(3) | còn: ['den', 'pee', 'tax']
Bước 3: lấy rex(2) | mới: - | queue: tad(3) tex(3) | còn: ['den', 'pee', 'tax']
Bước 4: lấy tad(3) | mới: ['tax'] | queue: tex(3) tax(4) | còn: ['den', 'pee']
Bước 5: lấy tex(3) | mới: - | queue: tax(4) | còn: ['den', 'pee']
Bước 6: lấy tax(4) == endWord -> return 4
Các dãy ngắn nhất: ['red -> ted -> tex -> tax', 'red -> ted -> tad -> tax', 'red -> rex -> tex -> tax']


In [38]:
# Đáp án Bài 2 và Bài 3
def answer_neighbors_in_set(word: str, word_set: set) -> list:
    result = []
    for i in range(len(word)):
        for c in string.ascii_lowercase:
            if c == word[i]:
                continue
            nxt = word[:i] + c + word[i + 1:]
            if nxt in word_set:
                result.append(nxt)
    return result


def answer_ladder_bfs_exercise(beginWord: str, endWord: str, wordList: List[str]) -> int:
    word_set = set(wordList)
    if endWord not in word_set:
        return 0
    word_set.discard(beginWord)
    q = deque([(beginWord, 1)])
    while q:
        word, level = q.popleft()
        if word == endWord:
            return level
        for nxt in answer_neighbors_in_set(word, word_set):
            word_set.remove(nxt)
            q.append((nxt, level + 1))
    return 0


S = {"hot", "dot", "dog", "lot", "log", "cog"}
assert answer_neighbors_in_set("hot", S) == ["dot", "lot"]
assert answer_neighbors_in_set("dog", S) == ["cog", "log", "dot"]
assert answer_neighbors_in_set("xyz", S) == []
assert answer_ladder_bfs_exercise(*EX1) == 5
assert answer_ladder_bfs_exercise(*EX2) == 0
assert answer_ladder_bfs_exercise("a", "c", ["a", "b", "c"]) == 2
assert answer_ladder_bfs_exercise("hot", "dog", ["hot", "dog"]) == 0
print("Bài 2, Bài 3: PASS")

Bài 2, Bài 3: PASS


### Đáp án Bài 4

Lưu ý tinh tế trong đáp án: `answer_neighbors_in_set` được gọi **trước** khi xóa các hàng xóm khỏi `word_set`, nên nó trả về list "chụp" tại thời điểm đó — an toàn để vừa duyệt list vừa xóa khỏi `set`. (Nếu bạn vừa duyệt **chính** một `set` vừa xóa phần tử của nó, Python sẽ báo `RuntimeError: Set changed size during iteration`.)

In [39]:
def answer_shortest_ladder_path(beginWord: str, endWord: str, wordList: List[str]) -> List[str]:
    word_set = set(wordList)
    if endWord not in word_set:
        return []
    word_set.discard(beginWord)
    parent = {beginWord: None}
    q = deque([beginWord])
    while q:
        word = q.popleft()
        if word == endWord:
            path = []
            cur = endWord
            while cur is not None:
                path.append(cur)
                cur = parent[cur]
            return path[::-1]
        for nxt in answer_neighbors_in_set(word, word_set):
            word_set.remove(nxt)
            parent[nxt] = word
            q.append(nxt)
    return []


p = answer_shortest_ladder_path(*EX1)
print(" -> ".join(p))
assert len(p) == 5 and is_valid_ladder(p, *EX1)
assert answer_shortest_ladder_path(*EX2) == []

rng = random.Random(4)
for _ in range(200):
    case = random_case(rng, 4, "abcde", 60)
    if case is None: continue
    p = answer_shortest_ladder_path(*case)
    assert len(p) == ladder_bfs_26(*case)
    assert p == [] or is_valid_ladder(p, *case)
print("Bài 4: PASS (cả 200 bộ ngẫu nhiên)")

hit -> hot -> dot -> dog -> cog
Bài 4: PASS (cả 200 bộ ngẫu nhiên)


In [40]:
# Đáp án Bài 5 và Bài 6
def answer_min_mutation(startGene: str, endGene: str, bank: List[str]) -> int:
    bank_set = set(bank)
    if endGene not in bank_set:
        return -1
    bank_set.discard(startGene)
    q = deque([(startGene, 0)])                 # đếm BƯỚC
    while q:
        gene, steps = q.popleft()
        if gene == endGene:
            return steps
        for i in range(len(gene)):
            for c in "ACGT":
                nxt = gene[:i] + c + gene[i + 1:]
                if nxt in bank_set:
                    bank_set.remove(nxt)
                    q.append((nxt, steps + 1))
    return -1


def answer_open_lock(deadends: List[str], target: str) -> int:
    dead = set(deadends)
    if "0000" in dead:
        return -1
    if target == "0000":
        return 0
    visited = {"0000"} | dead
    q = deque([("0000", 0)])
    while q:
        state, steps = q.popleft()
        for i in range(4):
            d = int(state[i])
            for nd in ((d + 1) % 10, (d - 1) % 10):
                nxt = state[:i] + str(nd) + state[i + 1:]
                if nxt == target:
                    return steps + 1
                if nxt not in visited:
                    visited.add(nxt)
                    q.append((nxt, steps + 1))
    return -1


assert answer_min_mutation("AACCGGTT", "AACCGGTA", ["AACCGGTA"]) == 1
assert answer_min_mutation("AACCGGTT", "AAACGGTA", ["AACCGGTA", "AACCGCTA", "AAACGGTA"]) == 2
assert answer_min_mutation("AACCGGTT", "AACCGGTA", []) == -1
assert answer_open_lock(["0201", "0101", "0102", "1212", "2002"], "0202") == 6
assert answer_open_lock(["8888"], "0009") == 1
assert answer_open_lock(["8887", "8889", "8878", "8898", "8788", "8988", "7888", "9888"], "8888") == -1
assert answer_open_lock(["0000"], "8888") == -1
assert answer_open_lock([], "0000") == 0
print("Bài 5, Bài 6: PASS")

Bài 5, Bài 6: PASS


## 18. Template cuối cùng nên nhớ

**BFS một chiều trên đồ thị ngầm** (dễ nhớ, đủ Accepted):

```python
from collections import deque

class Solution:
    def ladderLength(self, beginWord: str, endWord: str, wordList: List[str]) -> int:
        word_set = set(wordList)
        if endWord not in word_set:
            return 0
        q = deque([(beginWord, 1)])                  # đếm SỐ TỪ -> bắt đầu từ 1
        while q:
            word, level = q.popleft()
            if word == endWord:
                return level
            for i in range(len(word)):
                for c in "abcdefghijklmnopqrstuvwxyz":
                    nxt = word[:i] + c + word[i + 1:]
                    if nxt in word_set:
                        word_set.remove(nxt)         # visited lúc ĐƯA VÀO
                        q.append((nxt, level + 1))
        return 0
```

**BFS hai chiều** (nhanh nhất) — khác biệt cần nhớ:

```text
words = set(wordList); nếu endWord không có -> 0; bỏ begin, end khỏi words
front = {begin}, back = {end}, length = 1
while front and back:
    nếu front lớn hơn back -> đổi chỗ
    với mỗi nxt sinh từ front:
        nxt in back  -> return length + 1      # kiểm tra back TRƯỚC
        nxt in words -> xóa khỏi words, thêm vào next_front
    front = next_front; length += 1
return 0
```

**Câu thần chú**: *"Ngắn nhất, không trọng số → BFS. Hàng xóm sinh tại chỗ. Đánh dấu khi đưa vào. Biết cả hai đầu → gặp nhau ở giữa."*

## 19. Checklist tự kiểm tra trước khi submit

- Mình có trả `0` ngay khi `endWord` **không** có trong `wordList` chưa? (bắt buộc với BFS hai chiều)
- Mình đếm **số từ** (bắt đầu tầng `1`) chứ không phải số bước chưa?
- Từ điển đã đổi sang `set` trước khi tra `in` chưa?
- Queue dùng `deque.popleft()` (không phải `list.pop(0)`) chưa?
- Mình đánh dấu đã thăm (xóa khỏi `set` / thêm vào `visited`) **ngay khi đưa vào** queue chưa?
- Nếu `beginWord` có trong `wordList`, nó có bị thêm lại vào queue không? (`discard(beginWord)`)
- Với BFS hai chiều: có kiểm tra `nxt in back` **trước** `nxt in words`, luôn mở rộng phía nhỏ hơn, và trả `length + 1` chưa?
- Đã test: Example 1, 2; `"hit","hot",["hot"]` → 2; `"a","c",["a","b","c"]` → 2; `"hot","dog",["hot","dog"]` → 0 chưa?
- Mình có lỡ dùng DFS (hoặc DFS với `visited` vĩnh viễn) cho bài "ngắn nhất" không?